# نسخة قياس أداء Blockchain — Last.fm (العقد الموحّد مع Douban)

هذه النسخة مخصّصة لإعادة قياس **Last.fm** على المجتمعات الـ13 المعتمدة، مع استخدام **نفس عقد `AuthorizedNodeElection.sol` المستخدم في Douban حرفياً**.

- لا يتم تغيير نتائج التصويت أو قيم $A_k$.
- لا يتم تغيير المجتمعات أو عتبة الأغلبية أو مدة التأكيد.
- قياسات Gas وLatency وThroughput تبقى بنفس منهج القياس في نسخة Last.fm السابقة.
- تم استبدال عقد Last.fm السابق فقط بعقد Douban المرفوع، مع فحص SHA-256 وحجم الملف قبل النشر لضمان التطابق.
- SHA-256 للعقد الموحّد: `e61117cb56a2cde587a4e86e6e283352be09c5ed65d0d3c3c9c5cb473316a04e`.
- حجم ملف العقد: `20827` bytes.

**مهم:** ابدأ على Ganache جديدة، ثم أعد Wallet Mapping، وبعدها نفّذ الخلايا بالتسلسل حتى ملخص الأداء.


### Cell 0 — تثبيت المكتبات

In [1]:
import sys
import subprocess

packages = [
    "pandas",
    "openpyxl",
    "web3",
    "py-solc-x"
]

for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"[OK] {pkg} is installed")
    except ImportError:
        print(f"[INSTALL] Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

print("\n[OK] Required packages are ready.")

[OK] pandas is installed
[OK] openpyxl is installed
[OK] web3 is installed
[INSTALL] Installing py-solc-x ...

[OK] Required packages are ready.


### Cell 1  → فحص المسارات والملفات
### Cell 2  → تشغيل Step 1 لتحضير ملفات الإعلان
### Cell 3  → فحص نواتج Step 1
بعدها فقط ننتقل إلى
Step 2 الخاص بـ Ganache Wallet Mapping

### Cell 1 — فحص المسارات الأساسية

In [2]:
from pathlib import Path

VOTING_RUN_ID = "RUN_20260601_161913"

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

VOTING_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\2_التصويت"
) / VOTING_RUN_ID

VOTE_SCENARIO_NAME = "scenario1_global_friend_distance_sumdist_no_fallback"

INTEREST_MEMBERSHIP_CSV = (
    VOTING_ROOT
    / "outputs"
    / "step1_interest"
    / "lastfm_interest_membership.csv"
)

SUMMARY_CSV = (
    VOTING_ROOT
    / "outputs"
    / VOTE_SCENARIO_NAME
    / "final"
    / "lastfm_scenario1_global_friend_sumdist_no_fallback_summary.csv"
)

print("ANNOUNCEMENT_ROOT:")
print(ANNOUNCEMENT_ROOT)

print("\nVOTING_ROOT:")
print(VOTING_ROOT)

print("\nChecking files...")

print("Membership file exists:", INTEREST_MEMBERSHIP_CSV.exists())
print("Summary file exists   :", SUMMARY_CSV.exists())

if not INTEREST_MEMBERSHIP_CSV.exists():
    raise FileNotFoundError(INTEREST_MEMBERSHIP_CSV)

if not SUMMARY_CSV.exists():
    raise FileNotFoundError(SUMMARY_CSV)

print("\n[OK] Input files are ready.")

ANNOUNCEMENT_ROOT:
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان

VOTING_ROOT:
D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\2_التصويت\RUN_20260601_161913

Checking files...
Membership file exists: True
Summary file exists   : True

[OK] Input files are ready.


### Cell 2 — تثبيت أو فحص المكتبات

In [3]:
import sys
import subprocess

packages = ["pandas", "openpyxl"]

for pkg in packages:
    try:
        __import__(pkg)
        print(f"[OK] {pkg} is installed")
    except ImportError:
        print(f"[INSTALL] Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

print("\n[OK] Required packages are ready.")

[OK] pandas is installed
[OK] openpyxl is installed

[OK] Required packages are ready.


### Cell 2 — Step 1: تجهيز ملفات الإعلان

In [4]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Step 1 — Prepare Last.fm Announcement Files
# نسخة مستقلة بالكامل داخل Jupyter
# =====================================================================================

from pathlib import Path
from datetime import datetime
import json
import hashlib
import pandas as pd


# =====================================================================================
# 0. Paths
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

VOTING_RUN_ID = "RUN_20260601_161913"

VOTING_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\2_التصويت"
) / VOTING_RUN_ID

VOTE_SCENARIO_NAME = "scenario1_global_friend_distance_sumdist_no_fallback"

INTEREST_MEMBERSHIP_CSV = (
    VOTING_ROOT
    / "outputs"
    / "step1_interest"
    / "lastfm_interest_membership.csv"
)

VOTE_SCENARIO_DIR = (
    VOTING_ROOT
    / "outputs"
    / VOTE_SCENARIO_NAME
)

SUMMARY_CSV = (
    VOTE_SCENARIO_DIR
    / "final"
    / "lastfm_scenario1_global_friend_sumdist_no_fallback_summary.csv"
)

ALGORITHM_NAME = "Lastfm_Scenario1_GlobalNearest_FriendDistance_MinSum_NoFallback"


# =====================================================================================
# Final Evaluation Scope — 13 Common Communities
# =====================================================================================

FINAL_COMMUNITY_IDS = [
    28, 30, 38, 40, 44, 60, 99, 104, 107, 114, 204, 211, 506
]

EXPECTED_FINAL_COMMUNITIES = 13
EXPECTED_FINAL_MEMBER_ROWS = 673
EXPECTED_AUTHORIZATION_TRANSACTIONS = 347



# =====================================================================================
# 1. Output Folders and Files
# =====================================================================================

DATA_DIR = ANNOUNCEMENT_ROOT / "data"

COMMUNITIES_DIR = DATA_DIR / "communities"
CHAIN_ANNOUNCEMENTS_DIR = DATA_DIR / "chain_announcements"

OUTPUTS_DIR = ANNOUNCEMENT_ROOT / "outputs"
LOGS_DIR = OUTPUTS_DIR / "logs"

COMMUNITY_MEMBERS_OUT = COMMUNITIES_DIR / "community_members.csv"
AK_ANNOUNCEMENTS_OUT = CHAIN_ANNOUNCEMENTS_DIR / "ak_chain_announcements.csv"
VALIDATION_REPORT_OUT = LOGS_DIR / "validation_report.txt"


# =====================================================================================
# 2. Helper Functions
# =====================================================================================

def ensure_dirs():
    folders = [
        ANNOUNCEMENT_ROOT,
        DATA_DIR,
        COMMUNITIES_DIR,
        CHAIN_ANNOUNCEMENTS_DIR,
        OUTPUTS_DIR,
        LOGS_DIR,
    ]

    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)


def ensure_file_exists(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"[ERROR] Missing {label}: {path}")


def clean_int(value, field_name: str) -> int:
    if pd.isna(value):
        raise ValueError(f"[ERROR] Missing value in field: {field_name}")
    return int(value)


def clean_float(value, field_name: str) -> float:
    if pd.isna(value):
        raise ValueError(f"[ERROR] Missing value in field: {field_name}")
    return float(value)


def safe_bool(value) -> bool:
    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    text = str(value).strip().lower()

    if text in {"true", "1", "yes"}:
        return True

    if text in {"false", "0", "no"}:
        return False

    return bool(value)


def safe_str(value) -> str:
    if pd.isna(value):
        return ""
    return str(value)


def safe_int_or_none(value):
    if pd.isna(value):
        return None
    return int(value)


def safe_float_or_none(value):
    if pd.isna(value):
        return None
    return float(value)


def stable_json_hash(payload: dict) -> str:
    normalized = json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )

    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()


def file_sha256(path: Path):
    if not path.exists():
        return None

    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def get_detail_file_paths(community_id: int):
    nomination_file = (
        VOTE_SCENARIO_DIR
        / "nominations_per_interest"
        / f"lastfm_nominations_global_friend_comm_{community_id}.csv"
    )

    candidate_file = (
        VOTE_SCENARIO_DIR
        / "candidate_scores_per_interest"
        / f"lastfm_candidate_scores_comm_{community_id}.csv"
    )

    return nomination_file, candidate_file


# =====================================================================================
# 3. Build community_members.csv
# =====================================================================================

def build_community_members_file():
    ensure_file_exists(
        INTEREST_MEMBERSHIP_CSV,
        "lastfm_interest_membership.csv"
    )

    df = pd.read_csv(INTEREST_MEMBERSHIP_CSV)

    required_cols = {"interest_comm_id", "User"}
    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(
            f"[ERROR] lastfm_interest_membership.csv missing columns: {missing}\n"
            f"Found columns: {list(df.columns)}"
        )

    out = df.rename(
        columns={
            "User": "user_id",
            "interest_comm_id": "community_id"
        }
    )[["user_id", "community_id"]].copy()

    out = out.dropna(subset=["user_id", "community_id"])

    out["user_id"] = out["user_id"].astype(int)
    out["community_id"] = out["community_id"].astype(int)

    # Keep only the 13 communities used in the final Last.fm evaluation.
    out = out[out["community_id"].isin(FINAL_COMMUNITY_IDS)].copy()

    missing_scope = sorted(
        set(FINAL_COMMUNITY_IDS) - set(out["community_id"].unique().tolist())
    )
    if missing_scope:
        raise RuntimeError(
            f"[ERROR] Missing final-evaluation communities in membership file: {missing_scope}"
        )

    out = (
        out.drop_duplicates()
        .sort_values(["community_id", "user_id"])
        .reset_index(drop=True)
    )

    scope_communities = sorted(out["community_id"].unique().tolist())

    if len(scope_communities) != EXPECTED_FINAL_COMMUNITIES:
        raise RuntimeError(
            f"[ERROR] Expected {EXPECTED_FINAL_COMMUNITIES} final communities, "
            f"found {len(scope_communities)}: {scope_communities}"
        )

    if len(out) != EXPECTED_FINAL_MEMBER_ROWS:
        raise RuntimeError(
            f"[ERROR] Expected {EXPECTED_FINAL_MEMBER_ROWS} community-member rows "
            f"for the final 13-community scope, found {len(out)}."
        )

    out.to_csv(
        COMMUNITY_MEMBERS_OUT,
        index=False,
        encoding="utf-8-sig"
    )

    return out


# =====================================================================================
# 4. Build ak_chain_announcements.csv and JSON files
# =====================================================================================

def build_ak_announcement_files(community_members):
    ensure_file_exists(
        SUMMARY_CSV,
        "Last.fm voting summary CSV"
    )

    df_summary = pd.read_csv(SUMMARY_CSV)

    required_summary_cols = {
        "interest_comm_id",
        "interest_size",
        "nominating_voters",
        "abstain_count",
        "nomination_coverage",
        "unique_nominees",
        "leader",
        "leader_sum_dist",
        "leader_coverage_count",
        "leader_coverage_ratio",
        "leader_nominated_by_count",
        "leader_in_IC",
        "decision_basis",
        "tie_group_size",
    }

    missing = required_summary_cols - set(df_summary.columns)

    if missing:
        raise ValueError(
            f"[ERROR] Last.fm summary CSV missing columns: {missing}\n"
            f"Found columns: {list(df_summary.columns)}"
        )

    df_summary["interest_comm_id"] = df_summary["interest_comm_id"].astype(int)
    df_summary = df_summary[
        df_summary["interest_comm_id"].isin(FINAL_COMMUNITY_IDS)
    ].copy()

    missing_summary_scope = sorted(
        set(FINAL_COMMUNITY_IDS) - set(df_summary["interest_comm_id"].unique().tolist())
    )
    if missing_summary_scope:
        raise RuntimeError(
            f"[ERROR] Missing final-evaluation communities in voting summary: "
            f"{missing_summary_scope}"
        )

    community_sizes_from_members = (
        community_members
        .groupby("community_id")["user_id"]
        .nunique()
        .to_dict()
    )

    rows = []

    for _, r in df_summary.iterrows():
        community_id = clean_int(
            r["interest_comm_id"],
            "interest_comm_id"
        )

        if pd.isna(r["leader"]):
            print(f"[WARNING] Community {community_id} has no selected leader. Skipped.")
            continue

        A_k = clean_int(r["leader"], "leader")

        interest_size_summary = clean_int(
            r["interest_size"],
            "interest_size"
        )

        interest_size_members = int(
            community_sizes_from_members.get(community_id, 0)
        )

        community_size_used_on_chain = interest_size_members
        required_confirmations = (community_size_used_on_chain // 2) + 1

        nomination_file, candidate_file = get_detail_file_paths(community_id)

        nominations_hash = file_sha256(nomination_file)
        candidate_scores_hash = file_sha256(candidate_file)

        payload = {
            "dataset": "Last.fm",
            "algorithm": ALGORITHM_NAME,
            "voting_run_id": VOTING_RUN_ID,

            "community_id": community_id,
            "A_k": A_k,

            "interest_size_from_summary": interest_size_summary,
            "interest_size_from_membership_file": interest_size_members,
            "community_size_used_on_chain": community_size_used_on_chain,
            "required_confirmations": required_confirmations,

            "nominating_voters": clean_int(
                r["nominating_voters"],
                "nominating_voters"
            ),
            "abstain_count": clean_int(
                r["abstain_count"],
                "abstain_count"
            ),
            "nomination_coverage": clean_float(
                r["nomination_coverage"],
                "nomination_coverage"
            ),
            "unique_nominees": clean_int(
                r["unique_nominees"],
                "unique_nominees"
            ),

            "leader_sum_dist": clean_float(
                r["leader_sum_dist"],
                "leader_sum_dist"
            ),
            "leader_coverage_count": clean_int(
                r["leader_coverage_count"],
                "leader_coverage_count"
            ),
            "leader_coverage_ratio": clean_float(
                r["leader_coverage_ratio"],
                "leader_coverage_ratio"
            ),
            "leader_nominated_by_count": clean_int(
                r["leader_nominated_by_count"],
                "leader_nominated_by_count"
            ),
            "leader_in_IC": safe_bool(r["leader_in_IC"]),

            "decision_basis": safe_str(r["decision_basis"]),
            "tie_group_size": clean_int(
                r["tie_group_size"],
                "tie_group_size"
            ),

            "runner_up": safe_int_or_none(r["runner_up"]) if "runner_up" in df_summary.columns else None,
            "runner_up_sum_dist": safe_float_or_none(r["runner_up_sum_dist"]) if "runner_up_sum_dist" in df_summary.columns else None,
            "runner_up_coverage_count": safe_int_or_none(r["runner_up_coverage_count"]) if "runner_up_coverage_count" in df_summary.columns else None,
            "sum_margin": safe_float_or_none(r["sum_margin"]) if "sum_margin" in df_summary.columns else None,
            "coverage_margin": safe_int_or_none(r["coverage_margin"]) if "coverage_margin" in df_summary.columns else None,

            "nominations_sha256": nominations_hash,
            "candidate_scores_sha256": candidate_scores_hash,
        }

        result_hash = stable_json_hash(payload)

        payload_with_hash = {
            **payload,
            "result_hash": result_hash,
            "created_at": datetime.now().isoformat(timespec="seconds"),
        }

        json_path = (
            CHAIN_ANNOUNCEMENTS_DIR
            / f"lastfm_ak_announcement_comm_{community_id}.json"
        )

        json_path.write_text(
            json.dumps(
                payload_with_hash,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        rows.append({
            "community_id": community_id,
            "A_k": A_k,
            "result_hash": result_hash,

            "community_size_used_on_chain": community_size_used_on_chain,
            "required_confirmations": required_confirmations,

            "interest_size_from_summary": interest_size_summary,
            "interest_size_from_membership_file": interest_size_members,

            "nominating_voters": clean_int(
                r["nominating_voters"],
                "nominating_voters"
            ),
            "abstain_count": clean_int(
                r["abstain_count"],
                "abstain_count"
            ),
            "nomination_coverage": clean_float(
                r["nomination_coverage"],
                "nomination_coverage"
            ),

            "unique_nominees": clean_int(
                r["unique_nominees"],
                "unique_nominees"
            ),
            "leader_sum_dist": clean_float(
                r["leader_sum_dist"],
                "leader_sum_dist"
            ),
            "leader_coverage_count": clean_int(
                r["leader_coverage_count"],
                "leader_coverage_count"
            ),
            "leader_coverage_ratio": clean_float(
                r["leader_coverage_ratio"],
                "leader_coverage_ratio"
            ),
            "leader_nominated_by_count": clean_int(
                r["leader_nominated_by_count"],
                "leader_nominated_by_count"
            ),
            "leader_in_IC": safe_bool(r["leader_in_IC"]),

            "decision_basis": safe_str(r["decision_basis"]),
            "tie_group_size": clean_int(
                r["tie_group_size"],
                "tie_group_size"
            ),

            "json_file": str(json_path),
        })

    df_out = (
        pd.DataFrame(rows)
        .sort_values("community_id")
        .reset_index(drop=True)
    )

    if len(df_out) != EXPECTED_FINAL_COMMUNITIES:
        raise RuntimeError(
            f"[ERROR] Expected {EXPECTED_FINAL_COMMUNITIES} A_k announcements, "
            f"found {len(df_out)}."
        )

    threshold_sum = int(df_out["required_confirmations"].sum())
    if threshold_sum != EXPECTED_AUTHORIZATION_TRANSACTIONS:
        raise RuntimeError(
            f"[ERROR] Expected aggregate authorization threshold "
            f"{EXPECTED_AUTHORIZATION_TRANSACTIONS}, found {threshold_sum}."
        )

    df_out.to_csv(
        AK_ANNOUNCEMENTS_OUT,
        index=False,
        encoding="utf-8-sig"
    )

    return df_out


# =====================================================================================
# 5. Validation Report
# =====================================================================================

def write_validation_report(community_members, ak_announcements):
    lines = []

    lines.append("=" * 90)
    lines.append("Last.fm Announcement Stage — Step 1 Validation Report")
    lines.append("=" * 90)
    lines.append("")

    lines.append("[Input Paths]")
    lines.append(f"VOTING_ROOT             : {VOTING_ROOT}")
    lines.append(f"INTEREST_MEMBERSHIP_CSV : {INTEREST_MEMBERSHIP_CSV}")
    lines.append(f"SUMMARY_CSV             : {SUMMARY_CSV}")
    lines.append("")

    lines.append("[Output Root]")
    lines.append(f"ANNOUNCEMENT_ROOT       : {ANNOUNCEMENT_ROOT}")
    lines.append("")

    total_rows = len(community_members)
    total_unique_users = community_members["user_id"].nunique()
    total_communities = community_members["community_id"].nunique()

    lines.append("[Community Membership Summary]")
    lines.append(f"Total rows              : {total_rows}")
    lines.append(f"Total unique users      : {total_unique_users}")
    lines.append(f"Total communities       : {total_communities}")
    lines.append("")

    size_table = (
        community_members
        .groupby("community_id")["user_id"]
        .nunique()
        .reset_index(name="community_size")
        .sort_values("community_id")
    )

    lines.append("[Community Sizes]")
    lines.append(size_table.to_string(index=False))
    lines.append("")

    lines.append("[A_k Announcement Summary]")

    show_cols = [
        "community_id",
        "A_k",
        "community_size_used_on_chain",
        "required_confirmations",
        "decision_basis",
        "leader_in_IC",
    ]

    lines.append(
        ak_announcements[show_cols].to_string(index=False)
    )

    lines.append("")
    lines.append("[Methodological Notes]")
    lines.append("1) This step does not recompute A_k.")
    lines.append("2) A_k is read from the Last.fm voting summary file.")
    lines.append("3) result_hash links the off-chain voting result with the later on-chain announcement.")
    lines.append("4) required_confirmations = floor(community_size / 2) + 1.")
    lines.append("5) The selected A_k is not required to be a member of the same interest community.")
    lines.append("6) The announcer and confirmers will be validated later by the smart contract.")
    lines.append("7) This performance run is restricted to the 13 communities used in the final Last.fm evaluation.")

    VALIDATION_REPORT_OUT.write_text(
        "\n".join(lines),
        encoding="utf-8"
    )


# =====================================================================================
# 6. Run Step 1
# =====================================================================================

print("=" * 90)
print("Step 1 — Prepare Last.fm Announcement Files")
print("=" * 90)

ensure_dirs()

print("[CHECK] Membership file:", INTEREST_MEMBERSHIP_CSV)
print("[CHECK] Exists:", INTEREST_MEMBERSHIP_CSV.exists())

print("[CHECK] Summary file:", SUMMARY_CSV)
print("[CHECK] Exists:", SUMMARY_CSV.exists())

ensure_file_exists(
    INTEREST_MEMBERSHIP_CSV,
    "lastfm_interest_membership.csv"
)

ensure_file_exists(
    SUMMARY_CSV,
    "Last.fm voting summary CSV"
)

community_members = build_community_members_file()
ak_announcements = build_ak_announcement_files(community_members)

write_validation_report(
    community_members=community_members,
    ak_announcements=ak_announcements
)

print("\n[OK] Step 1 completed successfully.")
print("[OK] Created:", COMMUNITY_MEMBERS_OUT)
print("[OK] Created:", AK_ANNOUNCEMENTS_OUT)
print("[OK] Created:", VALIDATION_REPORT_OUT)

print("\n[SUMMARY]")
print("Total community-member rows:", len(community_members))
print("Total unique users:", community_members["user_id"].nunique())
print("Total communities:", community_members["community_id"].nunique())
print("Total A_k announcements:", len(ak_announcements))
print("Final community IDs:", sorted(ak_announcements["community_id"].tolist()))
print("Aggregate required confirmations:", int(ak_announcements["required_confirmations"].sum()))

if (
    len(ak_announcements) == EXPECTED_FINAL_COMMUNITIES
    and len(community_members) == EXPECTED_FINAL_MEMBER_ROWS
    and int(ak_announcements["required_confirmations"].sum()) == EXPECTED_AUTHORIZATION_TRANSACTIONS
):
    print("[FINAL SCOPE CHECK] PASS — 13 communities / 673 member rows / 347 authorization transactions.")
else:
    raise RuntimeError("[FINAL SCOPE CHECK] FAILED.")

print("\n[A_k SUMMARY]")
print(
    ak_announcements[
        [
            "community_id",
            "A_k",
            "community_size_used_on_chain",
            "required_confirmations",
            "decision_basis",
            "leader_in_IC",
        ]
    ].to_string(index=False)
)

Step 1 — Prepare Last.fm Announcement Files
[CHECK] Membership file: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\2_التصويت\RUN_20260601_161913\outputs\step1_interest\lastfm_interest_membership.csv
[CHECK] Exists: True
[CHECK] Summary file: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\2_التصويت\RUN_20260601_161913\outputs\scenario1_global_friend_distance_sumdist_no_fallback\final\lastfm_scenario1_global_friend_sumdist_no_fallback_summary.csv
[CHECK] Exists: True

[OK] Step 1 completed successfully.
[OK] Created: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\data\communities\community_members.csv
[OK] Created: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\data\chain_announcements\ak_chain_announcements.csv
[OK] Created: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\outputs\logs\validation_report.txt

[SUMMARY]
Total community-member rows: 673
Total unique users: 673
Total communities: 13
Total A_k announcements

### هنا يتم تشغيل الكناج بواسطة هذا الايعاز



ganache --wallet.totalAccounts 850 --chain.chainId 1337 --server.port 7545



### Cell 2 — Step 2: Build Wallet Mapping

In [5]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Step 2 — Build Wallet Mapping for Last.fm Announcement Stage
# =====================================================================================
#
# Purpose:
#   ربط كل user_id مطلوب بعنوان wallet من Ganache.
#
# Inputs:
#   1) community_members.csv
#   2) ak_chain_announcements.csv
#
# Outputs:
#   1) address_to_userid.csv
#   2) owner_address.txt
#   3) wallet_mapping_report.txt
#
# =====================================================================================

from pathlib import Path
import pandas as pd
from web3 import Web3


# =====================================================================================
# 0. Paths
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

DATA_DIR = ANNOUNCEMENT_ROOT / "data"

COMMUNITIES_DIR = DATA_DIR / "communities"
CHAIN_ANNOUNCEMENTS_DIR = DATA_DIR / "chain_announcements"
BLOCKCHAIN_DIR = DATA_DIR / "blockchain"

OUTPUTS_DIR = ANNOUNCEMENT_ROOT / "outputs"
LOGS_DIR = OUTPUTS_DIR / "logs"

COMMUNITY_MEMBERS_OUT = COMMUNITIES_DIR / "community_members.csv"
AK_ANNOUNCEMENTS_OUT = CHAIN_ANNOUNCEMENTS_DIR / "ak_chain_announcements.csv"

ADDRESS_TO_USERID_OUT = BLOCKCHAIN_DIR / "address_to_userid.csv"
OWNER_ADDRESS_OUT = BLOCKCHAIN_DIR / "owner_address.txt"
WALLET_MAPPING_REPORT_OUT = LOGS_DIR / "wallet_mapping_report.txt"

RPC_URL = "http://127.0.0.1:7545"


# =====================================================================================
# 1. Helper Functions
# =====================================================================================

def ensure_dirs():
    folders = [
        BLOCKCHAIN_DIR,
        OUTPUTS_DIR,
        LOGS_DIR,
    ]

    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)


def ensure_file_exists(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"[ERROR] Missing {label}: {path}")


def connect_ganache():
    w3 = Web3(Web3.HTTPProvider(RPC_URL))

    if not w3.is_connected():
        raise ConnectionError(
            f"[ERROR] Could not connect to Ganache at {RPC_URL}. "
            "Make sure Ganache is running."
        )

    return w3


# =====================================================================================
# 2. Run Step 2
# =====================================================================================

print("=" * 90)
print("Step 2 — Build Wallet Mapping")
print("=" * 90)

ensure_dirs()

ensure_file_exists(COMMUNITY_MEMBERS_OUT, "community_members.csv")
ensure_file_exists(AK_ANNOUNCEMENTS_OUT, "ak_chain_announcements.csv")

df_members = pd.read_csv(COMMUNITY_MEMBERS_OUT)
df_ak = pd.read_csv(AK_ANNOUNCEMENTS_OUT)

df_members["user_id"] = df_members["user_id"].astype(int)
df_members["community_id"] = df_members["community_id"].astype(int)

df_ak["community_id"] = df_ak["community_id"].astype(int)
df_ak["A_k"] = df_ak["A_k"].astype(int)

member_user_ids = set(
    df_members["user_id"]
    .dropna()
    .astype(int)
    .tolist()
)

ak_user_ids = set(
    df_ak["A_k"]
    .dropna()
    .astype(int)
    .tolist()
)

all_required_users = sorted(member_user_ids | ak_user_ids)
ak_not_in_membership = sorted(ak_user_ids - member_user_ids)

w3 = connect_ganache()
accounts = [Web3.to_checksum_address(a) for a in w3.eth.accounts]

print("[INFO] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Total Ganache accounts:", len(accounts))

if len(accounts) < 2:
    raise RuntimeError(
        "[ERROR] Ganache must contain at least 2 accounts."
    )

owner_address = accounts[0]
available_user_accounts = accounts[1:]

if len(available_user_accounts) < len(all_required_users):
    raise RuntimeError(
        "[ERROR] Not enough Ganache accounts.\n"
        f"Required user accounts: {len(all_required_users)}\n"
        f"Available user accounts: {len(available_user_accounts)}\n"
        f"Total Ganache accounts needed at least: {len(all_required_users) + 1}\n"
        "Restart Ganache with more accounts, for example:\n"
        "ganache --wallet.totalAccounts 850 --chain.chainId 1337 --server.port 7545"
    )

selected_user_accounts = available_user_accounts[:len(all_required_users)]

rows = []

for user_id, address in zip(all_required_users, selected_user_accounts):
    rows.append({
        "user_id": int(user_id),
        "address": address,
        "is_community_member": bool(user_id in member_user_ids),
        "is_Ak": bool(user_id in ak_user_ids),
    })

df_wallet = (
    pd.DataFrame(rows)
    .sort_values("user_id")
    .reset_index(drop=True)
)

df_wallet.to_csv(
    ADDRESS_TO_USERID_OUT,
    index=False,
    encoding="utf-8-sig"
)

OWNER_ADDRESS_OUT.write_text(
    owner_address,
    encoding="utf-8"
)

lines = []

lines.append("=" * 90)
lines.append("Last.fm Ganache Wallet Mapping Report")
lines.append("=" * 90)
lines.append("")
lines.append(f"RPC_URL                         : {RPC_URL}")
lines.append(f"Owner address                   : {owner_address}")
lines.append(f"Total Ganache accounts           : {len(accounts)}")
lines.append(f"Community member users           : {len(member_user_ids)}")
lines.append(f"Unique A_k users                 : {len(ak_user_ids)}")
lines.append(f"Total mapped users               : {len(all_required_users)}")
lines.append(f"A_k users not in community file  : {len(ak_not_in_membership)}")
lines.append("")
lines.append("A_k users not found in community_members.csv:")

if ak_not_in_membership:
    for u in ak_not_in_membership:
        lines.append(f"- {u}")
else:
    lines.append("None")

lines.append("")
lines.append(f"ADDRESS_TO_USERID_OUT            : {ADDRESS_TO_USERID_OUT}")
lines.append(f"OWNER_ADDRESS_OUT                : {OWNER_ADDRESS_OUT}")

WALLET_MAPPING_REPORT_OUT.write_text(
    "\n".join(lines),
    encoding="utf-8"
)

print("\n[OK] Step 2 completed successfully.")
print("[OK] Created:", ADDRESS_TO_USERID_OUT)
print("[OK] Created:", OWNER_ADDRESS_OUT)
print("[OK] Created:", WALLET_MAPPING_REPORT_OUT)

print("\n[SUMMARY]")
print("Owner address               :", owner_address)
print("Total Ganache accounts       :", len(accounts))
print("Community member users       :", len(member_user_ids))
print("Unique A_k users             :", len(ak_user_ids))
print("Total mapped users           :", len(all_required_users))
print("A_k not in community_members :", ak_not_in_membership)

print("\n[HEAD]")
print(df_wallet.head(10).to_string(index=False))

Step 2 — Build Wallet Mapping
[INFO] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Total Ganache accounts: 850

[OK] Step 2 completed successfully.
[OK] Created: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\data\blockchain\address_to_userid.csv
[OK] Created: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\data\blockchain\owner_address.txt
[OK] Created: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\outputs\logs\wallet_mapping_report.txt

[SUMMARY]
Owner address               : 0xC93B1093889dB280F60B4b0436c9f1a121D5D7f9
Total Ganache accounts       : 850
Community member users       : 673
Unique A_k users             : 13
Total mapped users           : 679
A_k not in community_members : [70, 189, 382, 585, 1694, 2043]

[HEAD]
 user_id                                    address  is_community_member  is_Ak
       2 0xd45a859586B8168cD32e0D6FdC194CEBB584b2c0                 True  False
       5 0xDBB28aE5EB21Ac677BA17382ecc4ec4E8C4

### إضافة قياسات أداء Blockchain — معتمدة لهذه الإعادة

شغّل هذه الخلية **بعد Wallet Mapping وقبل نشر العقد**. ستسجل Gas وLatency وبيانات الكتلة لكل معاملة، وتحفظ النتائج فوراً في `outputs/performance/PERF_...`.


In [6]:
# -*- coding: utf-8 -*-
# =====================================================================================
# PERFORMANCE INSTRUMENTATION — Last.fm
# قياسات إضافية فقط؛ لا تغيير في منطق العقد أو آلية الاختيار/التأكيد
# =====================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import time
import pandas as pd
from web3 import Web3

DATASET_NAME = "Last.fm"
ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)
RPC_URL = "http://127.0.0.1:7545"

FINAL_COMMUNITY_IDS = [
    28, 30, 38, 40, 44, 60, 99, 104, 107, 114, 204, 211, 506
]
EXPECTED_FINAL_COMMUNITIES = 13
EXPECTED_FINAL_MEMBER_ROWS = 673
EXPECTED_AUTHORIZATION_TRANSACTIONS = 347

PERFORMANCE_ROOT = ANNOUNCEMENT_ROOT / "outputs" / "performance"
PERFORMANCE_ROOT.mkdir(parents=True, exist_ok=True)

# إذا أُعيد تنفيذ هذه الخلية داخل نفس الـKernel لا ننشئ Run جديدًا بالخطأ.
if "PERF_RUN_ID" not in globals():
    PERF_RUN_ID = datetime.now().strftime("PERF_%Y%m%d_%H%M%S")

PERF_RUN_DIR = PERFORMANCE_ROOT / PERF_RUN_ID
PERF_RUN_DIR.mkdir(parents=True, exist_ok=True)

PERF_TX_CSV = PERF_RUN_DIR / "blockchain_performance_transactions.csv"
PERF_STAGE_SUMMARY_CSV = PERF_RUN_DIR / "blockchain_performance_by_stage.csv"
PERF_COMMUNITY_SUMMARY_CSV = PERF_RUN_DIR / "blockchain_performance_by_community.csv"
PERF_RUN_META_JSON = PERF_RUN_DIR / "run_metadata.json"
PERF_CURRENT_RUN_FILE = PERFORMANCE_ROOT / "current_performance_run.txt"

PERF_CURRENT_RUN_FILE.write_text(PERF_RUN_ID, encoding="utf-8")

PERF_COLUMNS = [
    "dataset", "run_id", "chain_id", "stage", "transaction_type",
    "community_id", "community_size", "required_confirmations",
    "batch_number", "batch_size", "confirmation_count_before",
    "confirmation_count_after", "is_finalizing", "stage_elapsed_sec",
    "sender_address", "contract_address", "tx_hash", "receipt_status",
    "block_number", "block_timestamp", "block_gas_used", "block_gas_limit",
    "tx_gas_limit", "gas_used", "effective_gas_price_wei",
    "effective_gas_price_gwei", "cost_wei", "cost_eth", "latency_sec",
    "submitted_at_utc", "receipt_at_utc", "note",
]


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def measure_transaction(w3: Web3, tx_sender, gas_limit: int) -> tuple:
    """Send one transaction, wait for its receipt, and return receipt + measured metrics."""
    submitted_at_utc = utc_now_iso()
    t0 = time.perf_counter()
    tx_hash = tx_sender()
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    t1 = time.perf_counter()
    receipt_at_utc = utc_now_iso()

    tx = w3.eth.get_transaction(tx_hash)
    block = w3.eth.get_block(receipt.blockNumber)
    gas_used = int(receipt.gasUsed)

    effective_gas_price = receipt.get("effectiveGasPrice", None)
    if effective_gas_price is None:
        effective_gas_price = tx.get("gasPrice", 0)
    effective_gas_price = int(effective_gas_price or 0)
    cost_wei = int(gas_used * effective_gas_price)

    metrics = {
        "chain_id": int(w3.eth.chain_id),
        "tx_hash": Web3.to_hex(tx_hash),
        "receipt_status": int(receipt.status),
        "block_number": int(receipt.blockNumber),
        "block_timestamp": int(block["timestamp"]),
        "block_gas_used": int(block.get("gasUsed", 0)),
        "block_gas_limit": int(block.get("gasLimit", 0)),
        "tx_gas_limit": int(gas_limit),
        "gas_used": gas_used,
        "effective_gas_price_wei": effective_gas_price,
        "effective_gas_price_gwei": effective_gas_price / 1e9,
        "cost_wei": cost_wei,
        "cost_eth": cost_wei / 1e18,
        "latency_sec": float(t1 - t0),
        "submitted_at_utc": submitted_at_utc,
        "receipt_at_utc": receipt_at_utc,
    }
    return receipt, metrics


def append_performance_row(**values) -> None:
    """Append one transaction row immediately, so partial measurements are not lost."""
    row = {col: None for col in PERF_COLUMNS}
    row.update({"dataset": DATASET_NAME, "run_id": PERF_RUN_ID})
    row.update(values)
    pd.DataFrame([row], columns=PERF_COLUMNS).to_csv(
        PERF_TX_CSV,
        mode="a",
        header=not PERF_TX_CSV.exists(),
        index=False,
        encoding="utf-8-sig",
    )


def update_run_metadata(**updates) -> None:
    meta = {}
    if PERF_RUN_META_JSON.exists():
        try:
            meta = json.loads(PERF_RUN_META_JSON.read_text(encoding="utf-8"))
        except Exception:
            meta = {}
    meta.update({
        "dataset": DATASET_NAME,
        "run_id": PERF_RUN_ID,
        "rpc_url": RPC_URL,
        "final_community_ids": FINAL_COMMUNITY_IDS,
        "expected_final_communities": EXPECTED_FINAL_COMMUNITIES,
        "expected_final_member_rows": EXPECTED_FINAL_MEMBER_ROWS,
        "expected_authorization_transactions": EXPECTED_AUTHORIZATION_TRANSACTIONS,
        "measurement_method": "Python Web3 submit-to-receipt timing using time.perf_counter; gas from transaction receipt.gasUsed",
        "measurement_scope": "Contract deployment, community-member loading, authorized-node announcement, member confirmations, and finalization-triggering confirmation",
        "note": "Local Ganache measurements; monetary cost uses Ganache effective gas price and is not a public-network fee estimate.",
    })
    meta.update(updates)
    PERF_RUN_META_JSON.write_text(
        json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
    )


w3_perf_check = Web3(Web3.HTTPProvider(RPC_URL))
update_run_metadata(
    started_at_utc=utc_now_iso(),
    connected=bool(w3_perf_check.is_connected()),
    chain_id=int(w3_perf_check.eth.chain_id) if w3_perf_check.is_connected() else None,
    starting_block=int(w3_perf_check.eth.block_number) if w3_perf_check.is_connected() else None,
    ganache_accounts=len(w3_perf_check.eth.accounts) if w3_perf_check.is_connected() else None,
)

print("[PERFORMANCE] Run ID:", PERF_RUN_ID)
print("[PERFORMANCE] Transaction log:", PERF_TX_CSV)
print("[PERFORMANCE] Metadata:", PERF_RUN_META_JSON)


[PERFORMANCE] Run ID: PERF_20260901_021204
[PERFORMANCE] Transaction log: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\outputs\performance\PERF_20260901_021204\blockchain_performance_transactions.csv
[PERFORMANCE] Metadata: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\outputs\performance\PERF_20260901_021204\run_metadata.json


### Cell 3 — كتابة ونشر العقد الذكي الموحّد

نفّذ `Cell 3-A` أولاً. هذه الخلية تكتب **نفس ملف عقد Douban حرفياً** وتتحقق من الحجم وSHA-256.

يجب أن يظهر في النهاية:

`[FINAL CHECK] PASS — Last.fm will deploy the exact same contract used in Douban.`

بعدها نفّذ `Cell 3-B` للنشر، ثم أكمل Member Loading → Announcement → Confirmations → Final Validation → Performance Summary.


In [7]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 3-A — Write EXACT Douban AuthorizedNodeElection.sol for Last.fm
# =====================================================================================
# This cell writes the exact smart-contract source used in the Douban performance run.
# Important: write_bytes() is used so Windows newline translation cannot alter the file.
# =====================================================================================

from pathlib import Path
import hashlib

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

CONTRACTS_DIR = ANNOUNCEMENT_ROOT / "contracts"
CONTRACT_FILE = CONTRACTS_DIR / "AuthorizedNodeElection.sol"
CONTRACTS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CONTRACT_SIZE = 20827
EXPECTED_CONTRACT_SHA256 = "e61117cb56a2cde587a4e86e6e283352be09c5ed65d0d3c3c9c5cb473316a04e"

solidity_code = r'''// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/*
    ========================================================================
    Contract Name: AuthorizedNodeElection
    ========================================================================

    الفكرة العامة:
    هذا العقد لا يحسب A_k.
    اختيار A_k يتم خارج السلسلة بواسطة Python.

    وظيفة العقد:
    1) تحميل أعضاء المجتمع.
    2) استقبال إعلان A_k.
    3) استقبال تأكيدات أعضاء المجتمع.
    4) تثبيت A_k عند الوصول إلى الأغلبية البسيطة.
    5) إلغاء الإعلان إذا انتهت مهلة التأكيد دون الوصول إلى الأغلبية.

    التصميم المعتمد:
    - إعلان واحد معلّق فقط لكل مجتمع.
    - لا يوجد snapshot.
    - A_k هو user_id وليس wallet address.
    - A_k لا يشترط أن يكون عضوًا في المجتمع.
    - المعلن والمؤكدون يجب أن يكونوا أعضاء في المجتمع.
*/

contract AuthorizedNodeElection {

    /*
        owner:
        هو الحساب الذي ينشر العقد.
        هذا الحساب فقط يستطيع تحميل أعضاء المجتمع.
    */
    address public owner;

    /*
        confirmationWindowSeconds:
        مدة السماح بالتأكيدات بعد إعلان A_k.

        مثال:
        300  = خمس دقائق
        3600 = ساعة واحدة

        هذه القيمة تُحدد عند نشر العقد من Remix.
    */
    uint256 public confirmationWindowSeconds;

    /*
        Announcement:
        يمثل حالة إعلان A_k لمجتمع معين.

        Ak:
        رقم المستخدم المختار كعقدة موثوقة ومخوّلة.

        resultHash:
        بصمة نتيجة الاختيار خارج السلسلة.

        confirmCount:
        عدد التأكيدات الصحيحة المقبولة.

        exists:
        هل يوجد إعلان معلّق؟

        finalized:
        هل تم تثبيت A_k نهائيًا؟

        announcer:
        عنوان العضو الذي أعلن A_k.

        announcedAt:
        وقت إنشاء الإعلان.

        deadline:
        آخر وقت مسموح به لقبول التأكيدات.
    */
    struct Announcement {
        uint256 Ak;
        bytes32 resultHash;
        uint256 confirmCount;
        bool exists;
        bool finalized;
        address announcer;
        uint256 announcedAt;
        uint256 deadline;
    }

    /*
        isMember[communityId][address] = true

        هذا يعني أن address عضو في المجتمع communityId.
    */
    mapping(uint256 => mapping(address => bool)) public isMember;

    /*
        communitySize[communityId]
        يخزن عدد أعضاء المجتمع.
    */
    mapping(uint256 => uint256) public communitySize;

    /*
        announcements[communityId]
        يخزن إعلان A_k للمجتمع.

        لا يوجد snapshot.
        لكل مجتمع إعلان واحد فقط في كل مرة.
    */
    mapping(uint256 => Announcement) public announcements;

    /*
        hasConfirmed[communityId][address]
        يمنع العضو من التأكيد أكثر من مرة.
    */
    mapping(uint256 => mapping(address => bool)) public hasConfirmed;

    /*
        confirmersList:
        نخزن المؤكدين حتى نستطيع تنظيف hasConfirmed
        عند إلغاء إعلان منتهي أو عند reset يدوي.
    */
    mapping(uint256 => address[]) private confirmersList;

    /*
        finalizedAk[communityId]
        يخزن القائد النهائي بعد التثبيت.
    */
    mapping(uint256 => uint256) public finalizedAk;

    /*
        Events:
        تستخدمها Python أو أي خدمة خارجية لمعرفة ما حدث داخل العقد.
    */
    event CommunityMembersLoaded(
        uint256 indexed communityId,
        uint256 loadedCount,
        uint256 totalCommunitySize
    );

    event AkAnnounced(
        uint256 indexed communityId,
        uint256 Ak,
        bytes32 resultHash,
        address indexed announcer,
        uint256 announcedAt,
        uint256 deadline
    );

    event AkConfirmed(
        uint256 indexed communityId,
        uint256 Ak,
        address indexed confirmer,
        uint256 confirmCount,
        uint256 requiredConfirmations
    );

    event AkFinalized(
        uint256 indexed communityId,
        uint256 Ak,
        bytes32 resultHash
    );

    event PendingAnnouncementExpired(
        uint256 indexed communityId,
        uint256 oldAk,
        bytes32 oldResultHash
    );

    event PendingAnnouncementReset(
        uint256 indexed communityId,
        uint256 oldAk,
        bytes32 oldResultHash
    );

    /*
        فقط owner يستطيع تنفيذ بعض الدوال الإدارية.
    */
    modifier onlyOwner() {
        require(msg.sender == owner, "Only owner can call this function");
        _;
    }

    /*
        فقط عضو المجتمع يستطيع الإعلان أو التأكيد.
    */
    modifier onlyCommunityMember(uint256 communityId) {
        require(
            isMember[communityId][msg.sender],
            "Caller is not a community member"
        );
        _;
    }

    /*
        constructor:
        يتم تنفيذه عند نشر العقد.

        _confirmationWindowSeconds:
        مدة التأكيد بالثواني.

        مثال للنشر من Remix:
        300  = خمس دقائق
        3600 = ساعة واحدة
    */
    constructor(uint256 _confirmationWindowSeconds) {
        require(
            _confirmationWindowSeconds > 0,
            "Confirmation window must be greater than zero"
        );

        owner = msg.sender;
        confirmationWindowSeconds = _confirmationWindowSeconds;
    }

    /*
        ====================================================================
        loadCommunityMembers
        ====================================================================

        الهدف:
        تحميل أعضاء مجتمع معين إلى العقد.

        ملاحظة:
        نحمّل wallet addresses وليس user_id.
    */
    function loadCommunityMembers(
        uint256 communityId,
        address[] calldata members
    ) external onlyOwner {
        require(members.length > 0, "Empty member list");

        uint256 added = 0;

        for (uint256 i = 0; i < members.length; i++) {
            address memberAddress = members[i];

            require(memberAddress != address(0), "Invalid member address");

            if (!isMember[communityId][memberAddress]) {
                isMember[communityId][memberAddress] = true;
                communitySize[communityId] += 1;
                added += 1;
            }
        }

        emit CommunityMembersLoaded(
            communityId,
            added,
            communitySize[communityId]
        );
    }

    /*
        ====================================================================
        requiredConfirmations
        ====================================================================

        الأغلبية البسيطة:
        floor(communitySize / 2) + 1

        مثال:
        104 عضو → 53 تأكيد.
    */
    function requiredConfirmations(
        uint256 communityId
    ) public view returns (uint256) {
        uint256 n = communitySize[communityId];

        require(n > 0, "Unknown or empty community");

        return (n / 2) + 1;
    }

    /*
        ====================================================================
        announceAk
        ====================================================================

        الهدف:
        إعلان A_k لمجتمع معين.

        الشروط:
        - المعلن يجب أن يكون عضوًا في المجتمع.
        - لا يوجد إعلان معلّق حاليًا.
        - لا يوجد A_k مثبت سابقًا.
        - resultHash يجب ألا يكون صفرًا.

        مهم:
        لا نتحقق أن A_k عضو في المجتمع.
    */
    function announceAk(
        uint256 communityId,
        uint256 Ak,
        bytes32 resultHash
    ) external onlyCommunityMember(communityId) {
        require(resultHash != bytes32(0), "Invalid result hash");

        Announcement storage a = announcements[communityId];

        require(
            !a.finalized,
            "A_k already finalized for this community"
        );

        require(
            !a.exists,
            "Pending announcement already exists"
        );

        uint256 currentTime = block.timestamp;
        uint256 deadline = currentTime + confirmationWindowSeconds;

        a.Ak = Ak;
        a.resultHash = resultHash;
        a.confirmCount = 1;
        a.exists = true;
        a.finalized = false;
        a.announcer = msg.sender;
        a.announcedAt = currentTime;
        a.deadline = deadline;

        /*
            المعلن يُحتسب كتأكيد أول.
        */
        hasConfirmed[communityId][msg.sender] = true;
        confirmersList[communityId].push(msg.sender);

        emit AkAnnounced(
            communityId,
            Ak,
            resultHash,
            msg.sender,
            currentTime,
            deadline
        );

        emit AkConfirmed(
            communityId,
            Ak,
            msg.sender,
            a.confirmCount,
            requiredConfirmations(communityId)
        );

        if (a.confirmCount >= requiredConfirmations(communityId)) {
            _finalizeAk(communityId);
        }
    }

    /*
        ====================================================================
        confirmAk
        ====================================================================

        الهدف:
        تأكيد الإعلان الموجود.

        الشروط:
        - المؤكد يجب أن يكون عضوًا في المجتمع.
        - الإعلان يجب أن يكون موجودًا.
        - الإعلان لم يتم تثبيته بعد.
        - لم تنتهِ مهلة التأكيد.
        - قيمة Ak يجب أن تطابق الإعلان.
        - العضو لم يؤكد سابقًا.
    */
    function confirmAk(
        uint256 communityId,
        uint256 Ak
    ) external onlyCommunityMember(communityId) {
        Announcement storage a = announcements[communityId];

        require(a.exists, "No pending announcement");

        require(!a.finalized, "A_k already finalized");

        require(
            block.timestamp <= a.deadline,
            "Confirmation deadline passed"
        );

        require(a.Ak == Ak, "A_k mismatch");

        require(
            !hasConfirmed[communityId][msg.sender],
            "Already confirmed"
        );

        hasConfirmed[communityId][msg.sender] = true;
        confirmersList[communityId].push(msg.sender);

        a.confirmCount += 1;

        emit AkConfirmed(
            communityId,
            Ak,
            msg.sender,
            a.confirmCount,
            requiredConfirmations(communityId)
        );

        if (a.confirmCount >= requiredConfirmations(communityId)) {
            _finalizeAk(communityId);
        }
    }

    /*
        ====================================================================
        _finalizeAk
        ====================================================================

        الهدف:
        تثبيت A_k نهائيًا بعد تحقق الأغلبية.
    */
    function _finalizeAk(uint256 communityId) internal {
        Announcement storage a = announcements[communityId];

        require(a.exists, "No announcement to finalize");

        require(!a.finalized, "Already finalized");

        require(
            a.confirmCount >= requiredConfirmations(communityId),
            "Threshold not reached"
        );

        a.finalized = true;
        finalizedAk[communityId] = a.Ak;

        emit AkFinalized(
            communityId,
            a.Ak,
            a.resultHash
        );
    }

    /*
        ====================================================================
        isAnnouncementExpired
        ====================================================================

        الهدف:
        معرفة هل الإعلان المعلق انتهت مهلته أم لا.

        يرجع true فقط إذا:
        - يوجد إعلان.
        - لم يتم تثبيته.
        - الوقت الحالي تجاوز deadline.
    */
    function isAnnouncementExpired(
        uint256 communityId
    ) public view returns (bool) {
        Announcement storage a = announcements[communityId];

        if (!a.exists) {
            return false;
        }

        if (a.finalized) {
            return false;
        }

        return block.timestamp > a.deadline;
    }

    /*
        ====================================================================
        expirePendingAnnouncement
        ====================================================================

        الهدف:
        إلغاء إعلان انتهت مهلته ولم يصل إلى الأغلبية.

        من يستطيع استدعاءها؟
        - owner
        - أو أي عضو من المجتمع

        لماذا نسمح للعضو؟
        حتى لا يبقى إعلان خبيث أو خاطئ معلقًا إذا انتهت مهلته.

        ملاحظة:
        الإلغاء لا يحدث تلقائيًا.
        يجب إرسال transaction لاستدعاء هذه الدالة بعد انتهاء الوقت.
    */
    function expirePendingAnnouncement(
        uint256 communityId
    ) external {
        require(
            msg.sender == owner || isMember[communityId][msg.sender],
            "Caller must be owner or community member"
        );

        Announcement storage a = announcements[communityId];

        require(a.exists, "No pending announcement");

        require(!a.finalized, "Cannot expire finalized A_k");

        require(
            block.timestamp > a.deadline,
            "Confirmation deadline has not passed"
        );

        uint256 oldAk = a.Ak;
        bytes32 oldResultHash = a.resultHash;

        _clearConfirmationFlags(communityId);

        delete announcements[communityId];

        emit PendingAnnouncementExpired(
            communityId,
            oldAk,
            oldResultHash
        );
    }

    /*
        ====================================================================
        resetPendingAnnouncement
        ====================================================================

        الهدف:
        إلغاء يدوي لإعلان معلّق قبل انتهاء الوقت.

        من يستدعيها؟
        owner فقط.

        تستخدم أثناء التجارب إذا تم إدخال إعلان خاطئ.
        لا تعمل بعد finalization.
    */
    function resetPendingAnnouncement(
        uint256 communityId
    ) external onlyOwner {
        Announcement storage a = announcements[communityId];

        require(a.exists, "No pending announcement");

        require(
            !a.finalized,
            "Cannot reset finalized A_k"
        );

        uint256 oldAk = a.Ak;
        bytes32 oldResultHash = a.resultHash;

        _clearConfirmationFlags(communityId);

        delete announcements[communityId];

        emit PendingAnnouncementReset(
            communityId,
            oldAk,
            oldResultHash
        );
    }

    /*
        ====================================================================
        _clearConfirmationFlags
        ====================================================================

        الهدف:
        تنظيف hasConfirmed عند إلغاء إعلان.

        السبب:
        إذا تم إلغاء إعلان، يجب أن يستطيع الأعضاء التأكيد مرة أخرى
        عند ظهور إعلان جديد.
    */
    function _clearConfirmationFlags(uint256 communityId) internal {
        address[] storage voters = confirmersList[communityId];

        for (uint256 i = 0; i < voters.length; i++) {
            hasConfirmed[communityId][voters[i]] = false;
        }

        delete confirmersList[communityId];
    }

    /*
        ====================================================================
        getAnnouncement
        ====================================================================

        الهدف:
        قراءة الحالة الأساسية لإعلان مجتمع معين.

        تم تقليل عدد القيم الراجعة لتجنب خطأ Stack too deep.
    */
    function getAnnouncement(
        uint256 communityId
    )
        external
        view
        returns (
            uint256 Ak,
            bytes32 resultHash,
            uint256 confirmCount,
            bool exists,
            bool finalized,
            address announcer
        )
    {
        Announcement storage a = announcements[communityId];

        return (
            a.Ak,
            a.resultHash,
            a.confirmCount,
            a.exists,
            a.finalized,
            a.announcer
        );
    }

    /*
        ====================================================================
        getAnnouncementTiming
        ====================================================================

        الهدف:
        قراءة وقت الإعلان، وقت انتهاء المهلة، وهل الإعلان منتهي أم لا.
    */
    function getAnnouncementTiming(
        uint256 communityId
    )
        external
        view
        returns (
            uint256 announcedAt,
            uint256 deadline,
            bool expired
        )
    {
        Announcement storage a = announcements[communityId];

        return (
            a.announcedAt,
            a.deadline,
            isAnnouncementExpired(communityId)
        );
    }

    /*
        ====================================================================
        getCommunityConfirmationInfo
        ====================================================================

        الهدف:
        قراءة حجم المجتمع، عدد التأكيدات المطلوبة، وعدد التأكيدات الحالية.
    */
    function getCommunityConfirmationInfo(
        uint256 communityId
    )
        external
        view
        returns (
            uint256 size,
            uint256 requiredCount,
            uint256 currentConfirmCount
        )
    {
        Announcement storage a = announcements[communityId];

        uint256 req = 0;

        if (communitySize[communityId] > 0) {
            req = requiredConfirmations(communityId);
        }

        return (
            communitySize[communityId],
            req,
            a.confirmCount
        );
    }

    /*
        ====================================================================
        hasAddressConfirmed
        ====================================================================

        الهدف:
        معرفة هل عنوان معين أكد إعلان المجتمع أم لا.
    */
    function hasAddressConfirmed(
        uint256 communityId,
        address account
    ) external view returns (bool) {
        return hasConfirmed[communityId][account];
    }

    /*
        ====================================================================
        getConfirmersCount
        ====================================================================

        الهدف:
        قراءة عدد العناوين التي أكدت.
    */
    function getConfirmersCount(
        uint256 communityId
    ) external view returns (uint256) {
        return confirmersList[communityId].length;
    }
}'''

# Reconstruct the exact Douban file bytes:
# - CRLF line endings
# - no blank line before the SPDX header
# - no trailing newline after the final closing brace
contract_bytes = solidity_code.replace("\n", "\r\n").encode("utf-8")
CONTRACT_FILE.write_bytes(contract_bytes)

actual_bytes = CONTRACT_FILE.read_bytes()
actual_size = len(actual_bytes)
actual_sha256 = hashlib.sha256(actual_bytes).hexdigest()

print("[OK] Unified Douban contract written for Last.fm.")
print("[PATH]", CONTRACT_FILE)
print("[VERIFY] File size:", actual_size, "bytes")
print("[VERIFY] SHA-256 :", actual_sha256)

if actual_size != EXPECTED_CONTRACT_SIZE:
    raise RuntimeError(
        f"Contract size mismatch: expected {EXPECTED_CONTRACT_SIZE}, got {actual_size}"
    )

if actual_sha256 != EXPECTED_CONTRACT_SHA256:
    raise RuntimeError(
        "Contract SHA-256 mismatch. The Last.fm contract is not byte-for-byte identical "
        "to the Douban contract."
    )

print("[FINAL CHECK] PASS — Last.fm will deploy the exact same contract used in Douban.")


[OK] Unified Douban contract written for Last.fm.
[PATH] D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\contracts\AuthorizedNodeElection.sol
[VERIFY] File size: 20827 bytes
[VERIFY] SHA-256 : e61117cb56a2cde587a4e86e6e283352be09c5ed65d0d3c3c9c5cb473316a04e
[FINAL CHECK] PASS — Last.fm will deploy the exact same contract used in Douban.


### Cell 3-B — Deploy Smart Contract

In [8]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 3-B — Deploy AuthorizedNodeElection Smart Contract
# =====================================================================================

from pathlib import Path
import json

from web3 import Web3
from solcx import compile_source, install_solc


# =====================================================================================
# Paths and Config
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

CONTRACTS_DIR = ANNOUNCEMENT_ROOT / "contracts"
CONTRACTS_BUILD_DIR = CONTRACTS_DIR / "build"

BLOCKCHAIN_DIR = ANNOUNCEMENT_ROOT / "data" / "blockchain"

CONTRACT_FILE = CONTRACTS_DIR / "AuthorizedNodeElection.sol"
ABI_OUT = CONTRACTS_BUILD_DIR / "AuthorizedNodeElection.abi.json"
ADDRESS_OUT = CONTRACTS_BUILD_DIR / "AuthorizedNodeElection.address.txt"
OWNER_ADDRESS_OUT = BLOCKCHAIN_DIR / "owner_address.txt"

RPC_URL = "http://127.0.0.1:7545"
SOLC_VERSION = "0.8.20"
CONFIRMATION_WINDOW_SECONDS = 3600
DEPLOY_TX_GAS = 7_000_000

CONTRACTS_BUILD_DIR.mkdir(parents=True, exist_ok=True)


# =====================================================================================
# Helper
# =====================================================================================

def ensure_file_exists(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"[ERROR] Missing {label}: {path}")


def connect_ganache():
    w3 = Web3(Web3.HTTPProvider(RPC_URL))

    if not w3.is_connected():
        raise ConnectionError(
            f"[ERROR] Could not connect to Ganache at {RPC_URL}. "
            "Make sure Ganache is running."
        )

    return w3


# =====================================================================================
# Deploy
# =====================================================================================

print("=" * 90)
print("Step 3 — Deploy AuthorizedNodeElection Smart Contract")
print("=" * 90)

ensure_file_exists(CONTRACT_FILE, "AuthorizedNodeElection.sol")
ensure_file_exists(OWNER_ADDRESS_OUT, "owner_address.txt")

w3 = connect_ganache()

print("[OK] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Accounts:", len(w3.eth.accounts))

owner_address = OWNER_ADDRESS_OUT.read_text(encoding="utf-8").strip()
owner_address = Web3.to_checksum_address(owner_address)

print("[INFO] Owner from file:", owner_address)
print("[INFO] Ganache account[0]:", w3.eth.accounts[0])

if owner_address.lower() != w3.eth.accounts[0].lower():
    raise RuntimeError(
        "Owner address in owner_address.txt does not match Ganache account[0]. "
        "Do not restart Ganache between Step 2 and Step 3. "
        "If Ganache was restarted, re-run Step 2."
    )

print(f"[INFO] Installing/using solc {SOLC_VERSION} ...")
install_solc(SOLC_VERSION)

source_code = CONTRACT_FILE.read_text(encoding="utf-8")

compiled = compile_source(
    source_code,
    output_values=["abi", "bin"],
    solc_version=SOLC_VERSION,
    optimize=True,
)

contract_id, contract_interface = compiled.popitem()

abi = contract_interface["abi"]
bytecode = contract_interface["bin"]

print("[OK] Contract compiled")
print("[INFO] Contract ID:", contract_id)

Contract = w3.eth.contract(
    abi=abi,
    bytecode=bytecode
)

receipt, perf = measure_transaction(
    w3,
    lambda: Contract.constructor(
        CONFIRMATION_WINDOW_SECONDS
    ).transact({
        "from": owner_address,
        "gas": DEPLOY_TX_GAS,
    }),
    gas_limit=DEPLOY_TX_GAS,
)

contract_address = receipt.contractAddress

append_performance_row(
    stage="deployment",
    transaction_type="Contract deployment",
    sender_address=owner_address,
    contract_address=contract_address,
    note="One-time deployment cost",
    **perf,
)

update_run_metadata(
    contract_address=str(contract_address),
    confirmation_window_seconds=int(CONFIRMATION_WINDOW_SECONDS),
    solidity_compiler="0.8.20",
    compiler_optimization=True,
)

ABI_OUT.write_text(
    json.dumps(abi, indent=2),
    encoding="utf-8"
)

ADDRESS_OUT.write_text(
    contract_address,
    encoding="utf-8"
)

print("\n[OK] Step 3 completed successfully.")
print("[OK] Contract deployed")
print("[INFO] Tx hash:", perf["tx_hash"])
print("[INFO] Contract address:", contract_address)
print("[PERFORMANCE] gasUsed:", perf["gas_used"])
print("[PERFORMANCE] latency_sec:", f'{perf["latency_sec"]:.6f}')
print("[OK] ABI saved to:", ABI_OUT)
print("[OK] Address saved to:", ADDRESS_OUT)

contract = w3.eth.contract(
    address=Web3.to_checksum_address(contract_address),
    abi=abi
)

print("\n[VERIFY]")
print("owner on-chain            :", contract.functions.owner().call())
print("confirmationWindowSeconds :", contract.functions.confirmationWindowSeconds().call())

Step 3 — Deploy AuthorizedNodeElection Smart Contract
[OK] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Accounts: 850
[INFO] Owner from file: 0xC93B1093889dB280F60B4b0436c9f1a121D5D7f9
[INFO] Ganache account[0]: 0xC93B1093889dB280F60B4b0436c9f1a121D5D7f9
[INFO] Installing/using solc 0.8.20 ...
[OK] Contract compiled
[INFO] Contract ID: <stdin>:AuthorizedNodeElection

[OK] Step 3 completed successfully.
[OK] Contract deployed
[INFO] Tx hash: 0x8fb9f29bc858abc08e352254fd85db6f8ed7384dfa8736499040735f17752af3
[INFO] Contract address: 0x81CC2969f72341fCC3DF58e920BA33CE8DE90f29
[PERFORMANCE] gasUsed: 1271695
[PERFORMANCE] latency_sec: 0.063346
[OK] ABI saved to: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\contracts\build\AuthorizedNodeElection.abi.json
[OK] Address saved to: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان\contracts\build\AuthorizedNodeElection.address.txt

[VERIFY]
owner on-chain            : 0xC93B1093889dB280F60B4b0436c9f1a121D5D

### Cell 4 — Step 4: Load Community Members

In [9]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 4 — Step 4: Load Last.fm Community Members to Smart Contract
# =====================================================================================

from pathlib import Path
import json
import pandas as pd
from web3 import Web3


# =====================================================================================
# Paths and Config
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

DATA_DIR = ANNOUNCEMENT_ROOT / "data"

COMMUNITIES_DIR = DATA_DIR / "communities"
BLOCKCHAIN_DIR = DATA_DIR / "blockchain"

CONTRACTS_DIR = ANNOUNCEMENT_ROOT / "contracts"
CONTRACTS_BUILD_DIR = CONTRACTS_DIR / "build"

OUTPUTS_DIR = ANNOUNCEMENT_ROOT / "outputs"
LOGS_DIR = OUTPUTS_DIR / "logs"

COMMUNITY_MEMBERS_OUT = COMMUNITIES_DIR / "community_members.csv"
ADDRESS_TO_USERID_OUT = BLOCKCHAIN_DIR / "address_to_userid.csv"
OWNER_ADDRESS_OUT = BLOCKCHAIN_DIR / "owner_address.txt"

ABI_OUT = CONTRACTS_BUILD_DIR / "AuthorizedNodeElection.abi.json"
ADDRESS_OUT = CONTRACTS_BUILD_DIR / "AuthorizedNodeElection.address.txt"

LOAD_REPORT_OUT = LOGS_DIR / "load_members_report.csv"

RPC_URL = "http://127.0.0.1:7545"

BATCH_SIZE = 50
LOAD_TX_GAS = 8_000_000


# =====================================================================================
# Helper Functions
# =====================================================================================

def ensure_file_exists(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"[ERROR] Missing {label}: {path}")


def connect_ganache():
    w3 = Web3(Web3.HTTPProvider(RPC_URL))

    if not w3.is_connected():
        raise ConnectionError(
            f"[ERROR] Could not connect to Ganache at {RPC_URL}. "
            "Make sure Ganache is running."
        )

    return w3


def load_contract(w3):
    ensure_file_exists(ABI_OUT, "AuthorizedNodeElection ABI")
    ensure_file_exists(ADDRESS_OUT, "AuthorizedNodeElection address")

    abi = json.loads(
        ABI_OUT.read_text(encoding="utf-8")
    )

    contract_address = Web3.to_checksum_address(
        ADDRESS_OUT.read_text(encoding="utf-8").strip()
    )

    return w3.eth.contract(
        address=contract_address,
        abi=abi
    )


# =====================================================================================
# Run Step 4
# =====================================================================================

print("=" * 90)
print("Step 4 — Load Last.fm Community Members to Smart Contract")
print("=" * 90)

LOGS_DIR.mkdir(parents=True, exist_ok=True)

ensure_file_exists(COMMUNITY_MEMBERS_OUT, "community_members.csv")
ensure_file_exists(ADDRESS_TO_USERID_OUT, "address_to_userid.csv")
ensure_file_exists(OWNER_ADDRESS_OUT, "owner_address.txt")
ensure_file_exists(ABI_OUT, "AuthorizedNodeElection ABI")
ensure_file_exists(ADDRESS_OUT, "AuthorizedNodeElection address")

w3 = connect_ganache()
contract = load_contract(w3)

owner_address = Web3.to_checksum_address(
    OWNER_ADDRESS_OUT.read_text(encoding="utf-8").strip()
)

onchain_owner = contract.functions.owner().call()

print("[OK] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Accounts:", len(w3.eth.accounts))
print("[INFO] Owner from file:", owner_address)
print("[INFO] Owner on-chain :", onchain_owner)
print("[INFO] Contract address:", contract.address)

if owner_address.lower() != onchain_owner.lower():
    raise RuntimeError(
        "Owner mismatch. The owner_address.txt does not match contract owner. "
        "If Ganache was restarted, re-run Step 2 and Step 3."
    )

df_members = pd.read_csv(COMMUNITY_MEMBERS_OUT)
df_addr = pd.read_csv(ADDRESS_TO_USERID_OUT)

df_members["user_id"] = df_members["user_id"].astype(int)
df_members["community_id"] = df_members["community_id"].astype(int)

df_addr["user_id"] = df_addr["user_id"].astype(int)
df_addr["address"] = df_addr["address"].astype(str)

user_to_address = dict(
    zip(
        df_addr["user_id"].astype(int),
        df_addr["address"].astype(str)
    )
)

report_rows = []

community_ids = sorted(
    df_members["community_id"].unique().tolist()
)

print("\n[START] Loading Last.fm community members...")
print("[INFO] Number of communities:", len(community_ids))
print("[INFO] Communities:", community_ids)

for community_id in community_ids:
    user_ids = (
        df_members.loc[df_members["community_id"] == community_id, "user_id"]
        .dropna()
        .astype(int)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    missing_users = [
        u for u in user_ids
        if u not in user_to_address
    ]

    if missing_users:
        raise RuntimeError(
            f"Missing wallet addresses for community {community_id}: {missing_users[:10]}"
        )

    addresses = [
        Web3.to_checksum_address(user_to_address[u])
        for u in user_ids
    ]

    before_size = contract.functions.communitySize(
        int(community_id)
    ).call()

    print("\n" + "-" * 90)
    print(f"[COMMUNITY] {community_id}")
    print("[INFO] Expected users to load:", len(user_ids))
    print("[INFO] Current on-chain size before loading:", before_size)

    tx_count = 0
    status = "loaded"
    load_stage_t0 = time.perf_counter()
    expected_required = (len(user_ids) // 2) + 1

    # حماية من إعادة التشغيل غير المقصود
    if before_size == len(user_ids):
        print("[SKIP] Community members already loaded on-chain.")
        status = "already_loaded"

    else:
        for i in range(0, len(addresses), BATCH_SIZE):
            batch = addresses[i:i + BATCH_SIZE]
            batch_number = tx_count + 1

            receipt, perf = measure_transaction(
                w3,
                lambda batch=batch: contract.functions.loadCommunityMembers(
                    int(community_id),
                    batch
                ).transact({
                    "from": owner_address,
                    "gas": LOAD_TX_GAS
                }),
                gas_limit=LOAD_TX_GAS,
            )

            tx_count += 1
            stage_elapsed = time.perf_counter() - load_stage_t0

            append_performance_row(
                stage="member_loading",
                transaction_type="Community-member loading",
                community_id=int(community_id),
                community_size=int(len(user_ids)),
                required_confirmations=int(expected_required),
                batch_number=int(batch_number),
                batch_size=int(len(batch)),
                stage_elapsed_sec=float(stage_elapsed),
                sender_address=owner_address,
                contract_address=contract.address,
                note="Setup cost; kept separate from authorization/finalization cost",
                **perf,
            )

            print(
                f"[OK] Batch {tx_count} loaded | "
                f"batch_size={len(batch)} | tx={perf['tx_hash']} | "
                f"gasUsed={perf['gas_used']} | latency={perf['latency_sec']:.6f}s"
            )

    after_size = contract.functions.communitySize(
        int(community_id)
    ).call()

    required = contract.functions.requiredConfirmations(
        int(community_id)
    ).call()

    size_matches = int(after_size) == int(len(user_ids))

    print("[INFO] On-chain size after loading:", after_size)
    print("[INFO] Required confirmations:", required)
    print("[CHECK] Size matches expected:", size_matches)

    report_rows.append({
        "community_id": int(community_id),
        "expected_members": int(len(user_ids)),
        "onchain_size_before": int(before_size),
        "onchain_size_after": int(after_size),
        "required_confirmations": int(required),
        "transactions_sent": int(tx_count),
        "status": status,
        "size_matches_expected": bool(size_matches),
    })

df_load_report = pd.DataFrame(report_rows)

df_load_report.to_csv(
    LOAD_REPORT_OUT,
    index=False,
    encoding="utf-8-sig"
)

print("\n[OK] Step 4 completed successfully.")
print("[OK] Load report created:", LOAD_REPORT_OUT)

print("\n[SUMMARY]")
print(
    df_load_report[
        [
            "community_id",
            "expected_members",
            "onchain_size_before",
            "onchain_size_after",
            "required_confirmations",
            "transactions_sent",
            "status",
            "size_matches_expected",
        ]
    ].to_string(index=False)
)

if df_load_report["size_matches_expected"].all():
    print("\n[FINAL CHECK] All communities were loaded correctly.")
else:
    print("\n[WARNING] Some communities do not match expected size. Check load_members_report.csv.")

Step 4 — Load Last.fm Community Members to Smart Contract
[OK] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Accounts: 850
[INFO] Owner from file: 0xC93B1093889dB280F60B4b0436c9f1a121D5D7f9
[INFO] Owner on-chain : 0xC93B1093889dB280F60B4b0436c9f1a121D5D7f9
[INFO] Contract address: 0x81CC2969f72341fCC3DF58e920BA33CE8DE90f29

[START] Loading Last.fm community members...
[INFO] Number of communities: 13
[INFO] Communities: [28, 30, 38, 40, 44, 60, 99, 104, 107, 114, 204, 211, 506]

------------------------------------------------------------------------------------------
[COMMUNITY] 28
[INFO] Expected users to load: 38
[INFO] Current on-chain size before loading: 0
[OK] Batch 1 loaded | batch_size=38 | tx=0x4294dd38d25a86037dcc35002d7db46976478de3bc391a0fe490d17909bde923 | gasUsed=950760 | latency=0.336265s
[INFO] On-chain size after loading: 38
[INFO] Required confirmations: 20
[CHECK] Size matches expected: True

----------------------------------------------------------------------

### Cell 5 — Step 5: Announce A_k

هذه الخطوة تقوم بإعلان العقدة المختارة A_k لكل مجتمع على العقد الذكي.

المهم:

لا تغلق Ganache
لا تعيد تشغيل Ganache
لا تعيد نشر العقد

In [10]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 5 — Step 5: Announce Last.fm A_k for Each Community
# =====================================================================================
#
# Purpose:
#   إعلان A_k لكل مجتمع على العقد الذكي.
#
# Inputs:
#   1) ak_chain_announcements.csv
#   2) community_members.csv
#   3) address_to_userid.csv
#   4) AuthorizedNodeElection ABI
#   5) AuthorizedNodeElection address
#
# Output:
#   announce_ak_report.csv
#
# Important:
#   - A_k is NOT recomputed here.
#   - A_k is read from ak_chain_announcements.csv.
#   - The announcer must be a member of the corresponding community.
#   - The selected A_k itself is not required to be a member of the community.
#   - The announceAk transaction counts as the first confirmation.
# =====================================================================================

from pathlib import Path
import json
import pandas as pd
from web3 import Web3


# =====================================================================================
# 0. Paths and Config
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

DATA_DIR = ANNOUNCEMENT_ROOT / "data"

COMMUNITIES_DIR = DATA_DIR / "communities"
BLOCKCHAIN_DIR = DATA_DIR / "blockchain"
CHAIN_ANNOUNCEMENTS_DIR = DATA_DIR / "chain_announcements"

CONTRACTS_BUILD_DIR = ANNOUNCEMENT_ROOT / "contracts" / "build"

OUTPUTS_DIR = ANNOUNCEMENT_ROOT / "outputs"
LOGS_DIR = OUTPUTS_DIR / "logs"

COMMUNITY_MEMBERS_OUT = COMMUNITIES_DIR / "community_members.csv"
AK_ANNOUNCEMENTS_OUT = CHAIN_ANNOUNCEMENTS_DIR / "ak_chain_announcements.csv"
ADDRESS_TO_USERID_OUT = BLOCKCHAIN_DIR / "address_to_userid.csv"

ABI_OUT = CONTRACTS_BUILD_DIR / "AuthorizedNodeElection.abi.json"
ADDRESS_OUT = CONTRACTS_BUILD_DIR / "AuthorizedNodeElection.address.txt"

ANNOUNCE_REPORT_OUT = LOGS_DIR / "announce_ak_report.csv"

RPC_URL = "http://127.0.0.1:7545"
ANNOUNCE_TX_GAS = 700_000


# =====================================================================================
# 1. Helper Functions
# =====================================================================================

def ensure_file_exists(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"[ERROR] Missing {label}: {path}")


def connect_ganache():
    w3 = Web3(Web3.HTTPProvider(RPC_URL))

    if not w3.is_connected():
        raise ConnectionError(
            f"[ERROR] Could not connect to Ganache at {RPC_URL}. "
            "Make sure Ganache is running."
        )

    return w3


def load_contract(w3):
    ensure_file_exists(ABI_OUT, "AuthorizedNodeElection ABI")
    ensure_file_exists(ADDRESS_OUT, "AuthorizedNodeElection address")

    abi = json.loads(
        ABI_OUT.read_text(encoding="utf-8")
    )

    contract_address = Web3.to_checksum_address(
        ADDRESS_OUT.read_text(encoding="utf-8").strip()
    )

    return w3.eth.contract(
        address=contract_address,
        abi=abi
    )


def result_hash_to_bytes32(result_hash_hex: str) -> bytes:
    h = str(result_hash_hex).strip()

    if h.startswith("0x"):
        h = h[2:]

    if len(h) != 64:
        raise ValueError(
            f"Invalid result_hash length. Expected 64 hex chars, got {len(h)}"
        )

    return bytes.fromhex(h)


# =====================================================================================
# 2. Run Step 5
# =====================================================================================

print("=" * 90)
print("Step 5 — Announce Last.fm A_k for Each Community")
print("=" * 90)

LOGS_DIR.mkdir(parents=True, exist_ok=True)

ensure_file_exists(AK_ANNOUNCEMENTS_OUT, "ak_chain_announcements.csv")
ensure_file_exists(COMMUNITY_MEMBERS_OUT, "community_members.csv")
ensure_file_exists(ADDRESS_TO_USERID_OUT, "address_to_userid.csv")
ensure_file_exists(ABI_OUT, "AuthorizedNodeElection ABI")
ensure_file_exists(ADDRESS_OUT, "AuthorizedNodeElection address")

w3 = connect_ganache()
contract = load_contract(w3)

print("[OK] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Accounts:", len(w3.eth.accounts))
print("[INFO] Contract address:", contract.address)

df_ak = pd.read_csv(AK_ANNOUNCEMENTS_OUT)
df_members = pd.read_csv(COMMUNITY_MEMBERS_OUT)
df_addr = pd.read_csv(ADDRESS_TO_USERID_OUT)

df_ak["community_id"] = df_ak["community_id"].astype(int)
df_ak["A_k"] = df_ak["A_k"].astype(int)

df_members["community_id"] = df_members["community_id"].astype(int)
df_members["user_id"] = df_members["user_id"].astype(int)

df_addr["user_id"] = df_addr["user_id"].astype(int)
df_addr["address"] = df_addr["address"].astype(str)

user_to_address = dict(
    zip(
        df_addr["user_id"].astype(int),
        df_addr["address"].astype(str)
    )
)


def select_announcer_for_community(community_id: int):
    """
    Selects the first community member as the announcer.
    The announcer must be a valid member of the community on-chain.
    """
    user_ids = (
        df_members.loc[df_members["community_id"] == community_id, "user_id"]
        .dropna()
        .astype(int)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    if not user_ids:
        raise RuntimeError(f"No members found for community {community_id}")

    announcer_user_id = int(user_ids[0])

    if announcer_user_id not in user_to_address:
        raise RuntimeError(
            f"Missing wallet address for announcer user_id={announcer_user_id}"
        )

    announcer_address = Web3.to_checksum_address(
        user_to_address[announcer_user_id]
    )

    return announcer_user_id, announcer_address


report_rows = []

print("\n[START] Announcing Last.fm A_k values...")

for _, row in df_ak.sort_values("community_id").iterrows():
    community_id = int(row["community_id"])
    A_k = int(row["A_k"])

    result_hash_hex = str(row["result_hash"])
    result_hash_bytes = result_hash_to_bytes32(result_hash_hex)

    announcer_user_id, announcer_address = select_announcer_for_community(
        community_id
    )

    print("\n" + "-" * 90)
    print(f"[COMMUNITY] {community_id}")
    print("[INFO] A_k:", A_k)
    print("[INFO] Announcer user_id:", announcer_user_id)
    print("[INFO] Announcer address:", announcer_address)

    is_member = contract.functions.isMember(
        int(community_id),
        announcer_address
    ).call()

    print("[INFO] Announcer is member on-chain:", is_member)

    if not is_member:
        raise RuntimeError(
            f"Selected announcer is not a member on-chain. "
            f"community={community_id}, user_id={announcer_user_id}"
        )

    before = contract.functions.getAnnouncement(
        int(community_id)
    ).call()

    before_exists = bool(before[3])
    before_finalized = bool(before[4])

    tx_hash_hex = None

    if before_finalized:
        print("[SKIP] A_k already finalized for this community.")
        status = "already_finalized"

    elif before_exists:
        print("[SKIP] Pending announcement already exists.")
        status = "already_pending"

    else:
        info_before_announce = contract.functions.getCommunityConfirmationInfo(
            int(community_id)
        ).call()
        community_size_before_announce = int(info_before_announce[0])
        required_before_announce = int(info_before_announce[1])
        confirm_before_announce = int(info_before_announce[2])

        receipt, perf = measure_transaction(
            w3,
            lambda: contract.functions.announceAk(
                int(community_id),
                int(A_k),
                result_hash_bytes
            ).transact({
                "from": announcer_address,
                "gas": ANNOUNCE_TX_GAS
            }),
            gas_limit=ANNOUNCE_TX_GAS,
        )

        tx_hash_hex = perf["tx_hash"]

        after_for_perf = contract.functions.getAnnouncement(
            int(community_id)
        ).call()
        confirm_after_announce = int(after_for_perf[2])
        finalized_after_announce = bool(after_for_perf[4])

        append_performance_row(
            stage="announcement",
            transaction_type="Authorized-node announcement",
            community_id=int(community_id),
            community_size=int(community_size_before_announce),
            required_confirmations=int(required_before_announce),
            confirmation_count_before=int(confirm_before_announce),
            confirmation_count_after=int(confirm_after_announce),
            is_finalizing=bool(finalized_after_announce),
            stage_elapsed_sec=float(perf["latency_sec"]),
            sender_address=announcer_address,
            contract_address=contract.address,
            note="Announcement is counted as the first confirmation by the implemented mechanism",
            **perf,
        )

        print("[OK] A_k announcement sent.")
        print("[INFO] Tx:", tx_hash_hex)
        print("[PERFORMANCE] gasUsed:", perf["gas_used"])
        print("[PERFORMANCE] latency_sec:", f'{perf["latency_sec"]:.6f}')

        status = "announced"

    after = contract.functions.getAnnouncement(
        int(community_id)
    ).call()

    timing = contract.functions.getAnnouncementTiming(
        int(community_id)
    ).call()

    info = contract.functions.getCommunityConfirmationInfo(
        int(community_id)
    ).call()

    after_Ak = int(after[0])
    after_result_hash = after[1].hex()
    after_confirm_count = int(after[2])
    after_exists = bool(after[3])
    after_finalized = bool(after[4])
    after_announcer = after[5]

    announced_at = int(timing[0])
    deadline = int(timing[1])
    expired = bool(timing[2])

    onchain_size = int(info[0])
    required_count = int(info[1])

    Ak_matches = after_Ak == A_k

    print("[STATE]")
    print("exists       :", after_exists)
    print("finalized    :", after_finalized)
    print("A_k on-chain :", after_Ak)
    print("A_k matches  :", Ak_matches)
    print("confirmCount :", after_confirm_count)
    print("required     :", required_count)
    print("expired      :", expired)

    report_rows.append({
        "community_id": community_id,

        "A_k_expected": A_k,
        "A_k_onchain": after_Ak,
        "A_k_matches": bool(Ak_matches),

        "result_hash_expected": result_hash_hex,
        "result_hash_onchain": after_result_hash,

        "announcer_user_id": announcer_user_id,
        "announcer_address": announcer_address,
        "announcer_onchain": after_announcer,

        "status": status,
        "tx_hash": tx_hash_hex,

        "exists": after_exists,
        "finalized": after_finalized,
        "confirm_count": after_confirm_count,
        "required_confirmations": required_count,
        "community_size_onchain": onchain_size,

        "announced_at": announced_at,
        "deadline": deadline,
        "expired": expired,
    })

df_announce_report = pd.DataFrame(report_rows)

df_announce_report.to_csv(
    ANNOUNCE_REPORT_OUT,
    index=False,
    encoding="utf-8-sig"
)

print("\n[OK] Step 5 completed successfully.")
print("[OK] Announce report created:", ANNOUNCE_REPORT_OUT)

print("\n[SUMMARY]")
print(
    df_announce_report[
        [
            "community_id",
            "A_k_expected",
            "A_k_onchain",
            "A_k_matches",
            "status",
            "exists",
            "finalized",
            "confirm_count",
            "required_confirmations",
            "expired",
        ]
    ].to_string(index=False)
)

if df_announce_report["A_k_matches"].all() and df_announce_report["exists"].all():
    print("\n[FINAL CHECK] All A_k announcements were stored correctly.")
else:
    print("\n[WARNING] Some announcements need checking. See announce_ak_report.csv.")

Step 5 — Announce Last.fm A_k for Each Community
[OK] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Accounts: 850
[INFO] Contract address: 0x81CC2969f72341fCC3DF58e920BA33CE8DE90f29

[START] Announcing Last.fm A_k values...

------------------------------------------------------------------------------------------
[COMMUNITY] 28
[INFO] A_k: 1858
[INFO] Announcer user_id: 2
[INFO] Announcer address: 0xd45a859586B8168cD32e0D6FdC194CEBB584b2c0
[INFO] Announcer is member on-chain: True
[OK] A_k announcement sent.
[INFO] Tx: 0xc9d7cbded8aad5388d9d2e1c7eb5a02e2ad17f9c5ae43edc82a35c9a0e172077
[PERFORMANCE] gasUsed: 234331
[PERFORMANCE] latency_sec: 0.046987
[STATE]
exists       : True
finalized    : False
A_k on-chain : 1858
A_k matches  : True
confirmCount : 1
required     : 20
expired      : False

------------------------------------------------------------------------------------------
[COMMUNITY] 30
[INFO] A_k: 423
[INFO] Announcer user_id: 21
[INFO] Announcer address: 0x89EC034d08ff

### Cell 6 — Step 6: Confirm A_k
### Cell 6 — Step 6: Confirm A_k Until Finalization
هذه الخطوة ترسل تأكيدات أعضاء كل مجتمع إلى أن يصل عدد التأكيدات إلى:

required_confirmations

ونتيجة Step 5 عندك صحيحة لأن كل المجتمعات كانت announced و confirm_count = 1 و finalized = False، وهذا يعني أننا جاهزون للتأكيد.

In [11]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 6 — Step 6: Confirm Last.fm A_k Until Finalization
# =====================================================================================
#
# Purpose:
#   إرسال تأكيدات أعضاء المجتمع على A_k المعلنة سابقًا.
#
# Important:
#   - Step 5 جعل confirm_count = 1 لأن معاملة announceAk تُحسب كتأكيد أول.
#   - هذه الخطوة ترسل باقي التأكيدات حتى تصل إلى required_confirmations.
#   - عند الوصول إلى required_confirmations تصبح finalized = True.
#
# Input files:
#   1) community_members.csv
#   2) address_to_userid.csv
#   3) AuthorizedNodeElection ABI
#   4) AuthorizedNodeElection address
#
# Output:
#   confirm_ak_report.csv
#
# =====================================================================================

from pathlib import Path
import json
import pandas as pd
from web3 import Web3


# =====================================================================================
# 0. Paths and Config
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

DATA_DIR = ANNOUNCEMENT_ROOT / "data"

COMMUNITIES_DIR = DATA_DIR / "communities"
BLOCKCHAIN_DIR = DATA_DIR / "blockchain"

CONTRACTS_BUILD_DIR = ANNOUNCEMENT_ROOT / "contracts" / "build"

OUTPUTS_DIR = ANNOUNCEMENT_ROOT / "outputs"
LOGS_DIR = OUTPUTS_DIR / "logs"

COMMUNITY_MEMBERS_OUT = COMMUNITIES_DIR / "community_members.csv"
ADDRESS_TO_USERID_OUT = BLOCKCHAIN_DIR / "address_to_userid.csv"

ABI_OUT = CONTRACTS_BUILD_DIR / "AuthorizedNodeElection.abi.json"
ADDRESS_OUT = CONTRACTS_BUILD_DIR / "AuthorizedNodeElection.address.txt"

CONFIRM_REPORT_OUT = LOGS_DIR / "confirm_ak_report.csv"

RPC_URL = "http://127.0.0.1:7545"
CONFIRM_TX_GAS = 300_000


# =====================================================================================
# 1. Helper Functions
# =====================================================================================

def ensure_file_exists(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"[ERROR] Missing {label}: {path}")


def connect_ganache():
    w3 = Web3(Web3.HTTPProvider(RPC_URL))

    if not w3.is_connected():
        raise ConnectionError(
            f"[ERROR] Could not connect to Ganache at {RPC_URL}. "
            "Make sure Ganache is running."
        )

    return w3


def load_contract(w3):
    ensure_file_exists(ABI_OUT, "AuthorizedNodeElection ABI")
    ensure_file_exists(ADDRESS_OUT, "AuthorizedNodeElection address")

    abi = json.loads(
        ABI_OUT.read_text(encoding="utf-8")
    )

    contract_address = Web3.to_checksum_address(
        ADDRESS_OUT.read_text(encoding="utf-8").strip()
    )

    return w3.eth.contract(
        address=contract_address,
        abi=abi
    )


# =====================================================================================
# 2. Run Step 6
# =====================================================================================

print("=" * 90)
print("Step 6 — Confirm Last.fm A_k Until Finalization")
print("=" * 90)

LOGS_DIR.mkdir(parents=True, exist_ok=True)

ensure_file_exists(COMMUNITY_MEMBERS_OUT, "community_members.csv")
ensure_file_exists(ADDRESS_TO_USERID_OUT, "address_to_userid.csv")
ensure_file_exists(ABI_OUT, "AuthorizedNodeElection ABI")
ensure_file_exists(ADDRESS_OUT, "AuthorizedNodeElection address")

w3 = connect_ganache()
contract = load_contract(w3)

print("[OK] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Accounts:", len(w3.eth.accounts))
print("[INFO] Contract address:", contract.address)

df_members = pd.read_csv(COMMUNITY_MEMBERS_OUT)
df_addr = pd.read_csv(ADDRESS_TO_USERID_OUT)

df_members["community_id"] = df_members["community_id"].astype(int)
df_members["user_id"] = df_members["user_id"].astype(int)

df_addr["user_id"] = df_addr["user_id"].astype(int)
df_addr["address"] = df_addr["address"].astype(str)

user_to_address = dict(
    zip(
        df_addr["user_id"].astype(int),
        df_addr["address"].astype(str)
    )
)


def get_member_addresses_for_community(community_id: int):
    """
    Returns sorted community members as:
        [(user_id, wallet_address), ...]
    """
    user_ids = (
        df_members.loc[df_members["community_id"] == community_id, "user_id"]
        .dropna()
        .astype(int)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    out = []

    for u in user_ids:
        if u not in user_to_address:
            raise RuntimeError(
                f"Missing wallet address for user_id={u} in community={community_id}"
            )

        out.append(
            (
                int(u),
                Web3.to_checksum_address(user_to_address[u])
            )
        )

    return out


report_rows = []

community_ids = sorted(
    df_members["community_id"].unique().tolist()
)

print("\n[START] Confirming Last.fm A_k announcements...")
print("[INFO] Number of communities:", len(community_ids))
print("[INFO] Communities:", community_ids)

for community_id in community_ids:
    print("\n" + "-" * 90)
    print(f"[COMMUNITY] {community_id}")

    ann = contract.functions.getAnnouncement(
        int(community_id)
    ).call()

    timing = contract.functions.getAnnouncementTiming(
        int(community_id)
    ).call()

    info = contract.functions.getCommunityConfirmationInfo(
        int(community_id)
    ).call()

    A_k = int(ann[0])
    exists = bool(ann[3])
    finalized = bool(ann[4])
    expired = bool(timing[2])

    community_size = int(info[0])
    required_count = int(info[1])
    current_confirm_count = int(info[2])

    print("[STATE BEFORE]")
    print("exists       :", exists)
    print("finalized    :", finalized)
    print("A_k          :", A_k)
    print("communitySize:", community_size)
    print("confirmCount :", current_confirm_count)
    print("required     :", required_count)
    print("expired      :", expired)

    status = "not_processed"
    tx_sent = 0
    last_tx_hash = None

    if not exists:
        print("[SKIP] No pending announcement for this community.")
        status = "no_pending_announcement"

    elif finalized:
        print("[SKIP] Already finalized.")
        status = "already_finalized"

    elif expired:
        print("[SKIP] Announcement expired before confirmation.")
        status = "expired_before_confirmation"

    else:
        members = get_member_addresses_for_community(community_id)
        confirmation_stage_t0 = time.perf_counter()

        for user_id, addr in members:
            # Read latest state before every confirmation transaction
            ann_now = contract.functions.getAnnouncement(
                int(community_id)
            ).call()

            info_now = contract.functions.getCommunityConfirmationInfo(
                int(community_id)
            ).call()

            finalized_now = bool(ann_now[4])
            current_confirm_count = int(info_now[2])
            required_count = int(info_now[1])

            if finalized_now:
                print("[OK] Finalization already reached.")
                status = "finalized"
                break

            if current_confirm_count >= required_count:
                print("[OK] Required confirmation count reached.")
                status = "threshold_reached"
                break

            already_confirmed = contract.functions.hasAddressConfirmed(
                int(community_id),
                addr
            ).call()

            if already_confirmed:
                continue

            is_member = contract.functions.isMember(
                int(community_id),
                addr
            ).call()

            if not is_member:
                continue

            try:
                confirm_count_before_tx = int(current_confirm_count)

                receipt, perf = measure_transaction(
                    w3,
                    lambda: contract.functions.confirmAk(
                        int(community_id),
                        int(A_k)
                    ).transact({
                        "from": addr,
                        "gas": CONFIRM_TX_GAS
                    }),
                    gas_limit=CONFIRM_TX_GAS,
                )

                tx_sent += 1
                last_tx_hash = perf["tx_hash"]

                info_after_tx = contract.functions.getCommunityConfirmationInfo(
                    int(community_id)
                ).call()
                ann_after_tx = contract.functions.getAnnouncement(
                    int(community_id)
                ).call()

                confirm_count_after_tx = int(info_after_tx[2])
                finalized_after_tx = bool(ann_after_tx[4])
                stage_elapsed = time.perf_counter() - confirmation_stage_t0

                append_performance_row(
                    stage="confirmation",
                    transaction_type="Member confirmation",
                    community_id=int(community_id),
                    community_size=int(community_size),
                    required_confirmations=int(required_count),
                    confirmation_count_before=int(confirm_count_before_tx),
                    confirmation_count_after=int(confirm_count_after_tx),
                    is_finalizing=bool(finalized_after_tx),
                    stage_elapsed_sec=float(stage_elapsed),
                    sender_address=addr,
                    contract_address=contract.address,
                    note=(
                        "Finalization-triggering confirmation"
                        if finalized_after_tx
                        else "Regular member confirmation"
                    ),
                    **perf,
                )

                print(
                    f"[OK] confirmation | "
                    f"user_id={user_id} | "
                    f"count={confirm_count_after_tx}/{required_count} | "
                    f"tx={last_tx_hash} | gasUsed={perf['gas_used']} | "
                    f"latency={perf['latency_sec']:.6f}s | "
                    f"finalizing={finalized_after_tx}"
                )

            except Exception as e:
                print(
                    f"[WARNING] confirm failed | "
                    f"user_id={user_id} | "
                    f"error={str(e)}"
                )

        # Final state after finishing attempts for this community
        ann_final = contract.functions.getAnnouncement(
            int(community_id)
        ).call()

        info_final = contract.functions.getCommunityConfirmationInfo(
            int(community_id)
        ).call()

        finalized_final = bool(ann_final[4])
        current_confirm_count_final = int(info_final[2])
        required_count_final = int(info_final[1])

        if finalized_final:
            status = "finalized"
        elif current_confirm_count_final >= required_count_final:
            status = "threshold_reached_but_check_needed"
        else:
            status = "not_enough_confirmations"

    # Read final state for report
    ann_after = contract.functions.getAnnouncement(
        int(community_id)
    ).call()

    timing_after = contract.functions.getAnnouncementTiming(
        int(community_id)
    ).call()

    info_after = contract.functions.getCommunityConfirmationInfo(
        int(community_id)
    ).call()

    final_Ak = int(ann_after[0])
    final_confirm_count = int(ann_after[2])
    final_exists = bool(ann_after[3])
    final_finalized = bool(ann_after[4])
    final_expired = bool(timing_after[2])

    final_size = int(info_after[0])
    final_required = int(info_after[1])

    finalizedAk_value = contract.functions.finalizedAk(
        int(community_id)
    ).call()

    print("[STATE AFTER]")
    print("exists       :", final_exists)
    print("finalized    :", final_finalized)
    print("A_k          :", final_Ak)
    print("confirmCount :", final_confirm_count)
    print("required     :", final_required)
    print("finalizedAk  :", finalizedAk_value)
    print("expired      :", final_expired)
    print("tx_sent      :", tx_sent)
    print("status       :", status)

    report_rows.append({
        "community_id": int(community_id),
        "A_k": int(final_Ak),

        "status": status,
        "tx_sent": int(tx_sent),
        "last_tx_hash": last_tx_hash,

        "exists": final_exists,
        "finalized": final_finalized,
        "confirm_count": final_confirm_count,
        "required_confirmations": final_required,

        "community_size": final_size,
        "finalizedAk": int(finalizedAk_value),
        "expired": final_expired,
        "deadline": int(timing_after[1]),
    })

df_confirm_report = pd.DataFrame(report_rows)

df_confirm_report.to_csv(
    CONFIRM_REPORT_OUT,
    index=False,
    encoding="utf-8-sig"
)

print("\n[OK] Step 6 completed.")
print("[OK] Confirm report created:", CONFIRM_REPORT_OUT)

print("\n[SUMMARY]")
print(
    df_confirm_report[
        [
            "community_id",
            "A_k",
            "status",
            "finalized",
            "confirm_count",
            "required_confirmations",
            "finalizedAk",
            "expired",
            "tx_sent",
        ]
    ].to_string(index=False)
)

if df_confirm_report["finalized"].all():
    print("\n[FINAL CHECK] All A_k announcements were finalized successfully.")
else:
    print("\n[WARNING] Some communities were not finalized. Check confirm_ak_report.csv.")

Step 6 — Confirm Last.fm A_k Until Finalization
[OK] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Accounts: 850
[INFO] Contract address: 0x81CC2969f72341fCC3DF58e920BA33CE8DE90f29

[START] Confirming Last.fm A_k announcements...
[INFO] Number of communities: 13
[INFO] Communities: [28, 30, 38, 40, 44, 60, 99, 104, 107, 114, 204, 211, 506]

------------------------------------------------------------------------------------------
[COMMUNITY] 28
[STATE BEFORE]
exists       : True
finalized    : False
A_k          : 1858
communitySize: 38
confirmCount : 1
required     : 20
expired      : False
[OK] confirmation | user_id=63 | count=2/20 | tx=0x6016642a73c82e704b1caf3161632b621e4cb31818882dba0293bbcbe44ac3cd | gasUsed=90844 | latency=0.034952s | finalizing=False
[OK] confirmation | user_id=66 | count=3/20 | tx=0xfcfc315abf8b330e2c0ab60fbf341ddc5865c632606ecb8bbc20b492a9dc2313 | gasUsed=90844 | latency=0.029944s | finalizing=False
[OK] confirmation | user_id=96 | count=4/20 | tx=0x2a82

### Cell 7 — Step 7: Read and Validate Finalized A_k

هذه الخطوة تقرأ القيم النهائية من العقد الذكي وتتحقق من:

expected_Ak == onchain_Ak == finalized_Ak
confirm_count == required_confirmations
finalized == True
expired == False
hash_matches == True
validation_status == valid_finalized

بما أن Step 6 نجح وكل المجتمعات أصبحت finalized=True، هذه الخطوة يجب أن تكون قراءة تحقق نهائية فقط.

In [12]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Cell 7 — Step 7: Read and Validate Finalized Last.fm A_k
# =====================================================================================
#
# Purpose:
#   قراءة A_k النهائية من العقد الذكي والتحقق من مطابقتها مع نتائج التصويت off-chain.
#
# Checks:
#   1) expected_Ak == onchain_Ak == finalized_Ak
#   2) finalized == True
#   3) confirm_count == required_confirmations
#   4) expired == False
#   5) result_hash on-chain matches result_hash off-chain
#
# Output:
#   finalized_ak_report.csv
#
# =====================================================================================

from pathlib import Path
import json
import pandas as pd
from web3 import Web3


# =====================================================================================
# 0. Paths and Config
# =====================================================================================

ANNOUNCEMENT_ROOT = Path(
    r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\3_الاعلان"
)

DATA_DIR = ANNOUNCEMENT_ROOT / "data"

CHAIN_ANNOUNCEMENTS_DIR = DATA_DIR / "chain_announcements"

CONTRACTS_BUILD_DIR = ANNOUNCEMENT_ROOT / "contracts" / "build"

OUTPUTS_DIR = ANNOUNCEMENT_ROOT / "outputs"
LOGS_DIR = OUTPUTS_DIR / "logs"

AK_ANNOUNCEMENTS_OUT = CHAIN_ANNOUNCEMENTS_DIR / "ak_chain_announcements.csv"

ABI_OUT = CONTRACTS_BUILD_DIR / "AuthorizedNodeElection.abi.json"
ADDRESS_OUT = CONTRACTS_BUILD_DIR / "AuthorizedNodeElection.address.txt"

FINALIZED_REPORT_OUT = LOGS_DIR / "finalized_ak_report.csv"

RPC_URL = "http://127.0.0.1:7545"


# =====================================================================================
# 1. Helper Functions
# =====================================================================================

def ensure_file_exists(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"[ERROR] Missing {label}: {path}")


def connect_ganache():
    w3 = Web3(Web3.HTTPProvider(RPC_URL))

    if not w3.is_connected():
        raise ConnectionError(
            f"[ERROR] Could not connect to Ganache at {RPC_URL}. "
            "Make sure Ganache is running."
        )

    return w3


def load_contract(w3):
    ensure_file_exists(ABI_OUT, "AuthorizedNodeElection ABI")
    ensure_file_exists(ADDRESS_OUT, "AuthorizedNodeElection address")

    abi = json.loads(
        ABI_OUT.read_text(encoding="utf-8")
    )

    contract_address = Web3.to_checksum_address(
        ADDRESS_OUT.read_text(encoding="utf-8").strip()
    )

    return w3.eth.contract(
        address=contract_address,
        abi=abi
    )


# =====================================================================================
# 2. Run Step 7
# =====================================================================================

print("=" * 90)
print("Step 7 — Read and Validate Finalized Last.fm A_k")
print("=" * 90)

LOGS_DIR.mkdir(parents=True, exist_ok=True)

ensure_file_exists(AK_ANNOUNCEMENTS_OUT, "ak_chain_announcements.csv")
ensure_file_exists(ABI_OUT, "AuthorizedNodeElection ABI")
ensure_file_exists(ADDRESS_OUT, "AuthorizedNodeElection address")

w3 = connect_ganache()
contract = load_contract(w3)

print("[OK] Connected to Ganache")
print("[INFO] Chain ID:", w3.eth.chain_id)
print("[INFO] Accounts:", len(w3.eth.accounts))
print("[INFO] Contract address:", contract.address)

df_expected = pd.read_csv(AK_ANNOUNCEMENTS_OUT)

df_expected["community_id"] = df_expected["community_id"].astype(int)
df_expected["A_k"] = df_expected["A_k"].astype(int)

community_ids = sorted(
    df_expected["community_id"].unique().tolist()
)

rows = []

print("\n[START] Reading finalized Last.fm A_k values from contract...")
print("[INFO] Number of communities:", len(community_ids))
print("[INFO] Communities:", community_ids)

for community_id in community_ids:
    expected_row = df_expected[
        df_expected["community_id"] == community_id
    ].iloc[0]

    expected_Ak = int(expected_row["A_k"])
    expected_required = int(expected_row["required_confirmations"])
    expected_hash = str(expected_row["result_hash"]).strip()

    ann = contract.functions.getAnnouncement(
        int(community_id)
    ).call()

    timing = contract.functions.getAnnouncementTiming(
        int(community_id)
    ).call()

    info = contract.functions.getCommunityConfirmationInfo(
        int(community_id)
    ).call()

    finalized_Ak = int(
        contract.functions.finalizedAk(
            int(community_id)
        ).call()
    )

    onchain_Ak = int(ann[0])
    onchain_result_hash = ann[1].hex()
    confirm_count = int(ann[2])
    exists = bool(ann[3])
    finalized = bool(ann[4])
    announcer = ann[5]

    announced_at = int(timing[0])
    deadline = int(timing[1])
    expired = bool(timing[2])

    community_size = int(info[0])
    required_count = int(info[1])

    Ak_matches = (
        expected_Ak == onchain_Ak == finalized_Ak
    )

    required_matches = (
        expected_required == required_count
    )

    finalized_ok = (
        finalized and confirm_count >= required_count
    )

    confirm_count_matches = (
        confirm_count == required_count
    )

    hash_matches = (
        expected_hash.lower().replace("0x", "")
        ==
        onchain_result_hash.lower().replace("0x", "")
    )

    if (
        exists
        and finalized_ok
        and confirm_count_matches
        and Ak_matches
        and required_matches
        and hash_matches
        and not expired
    ):
        validation_status = "valid_finalized"
    else:
        validation_status = "check_needed"

    print("\n" + "-" * 90)
    print(f"[COMMUNITY] {community_id}")
    print("expected_Ak           :", expected_Ak)
    print("onchain_Ak            :", onchain_Ak)
    print("finalized_Ak          :", finalized_Ak)
    print("exists                :", exists)
    print("finalized             :", finalized)
    print("confirm_count         :", confirm_count)
    print("required_count        :", required_count)
    print("confirm_count_matches :", confirm_count_matches)
    print("expired               :", expired)
    print("Ak_matches            :", Ak_matches)
    print("required_matches      :", required_matches)
    print("hash_matches          :", hash_matches)
    print("validation_status     :", validation_status)

    rows.append({
        "community_id": int(community_id),

        "expected_Ak": expected_Ak,
        "onchain_Ak": onchain_Ak,
        "finalized_Ak": finalized_Ak,

        "exists": exists,
        "finalized": finalized,
        "expired": expired,

        "community_size": community_size,
        "confirm_count": confirm_count,
        "required_confirmations": required_count,
        "expected_required_confirmations": expected_required,
        "confirm_count_matches": confirm_count_matches,

        "announcer_address": announcer,

        "expected_result_hash": expected_hash,
        "onchain_result_hash": onchain_result_hash,

        "Ak_matches": Ak_matches,
        "required_matches": required_matches,
        "hash_matches": hash_matches,
        "finalized_ok": finalized_ok,

        "announced_at": announced_at,
        "deadline": deadline,

        "validation_status": validation_status,
    })

df_finalized_report = pd.DataFrame(rows)

df_finalized_report.to_csv(
    FINALIZED_REPORT_OUT,
    index=False,
    encoding="utf-8-sig"
)

print("\n[OK] Step 7 completed successfully.")
print("[OK] Finalized report created:", FINALIZED_REPORT_OUT)

print("\n[FINAL SUMMARY]")
print(
    df_finalized_report[
        [
            "community_id",
            "expected_Ak",
            "onchain_Ak",
            "finalized_Ak",
            "finalized",
            "confirm_count",
            "required_confirmations",
            "confirm_count_matches",
            "expired",
            "Ak_matches",
            "hash_matches",
            "validation_status",
        ]
    ].to_string(index=False)
)

if (df_finalized_report["validation_status"] == "valid_finalized").all():
    print("\n[FINAL CHECK] All finalized A_k values are valid and match off-chain results.")
else:
    print("\n[WARNING] Some finalized results need checking. See finalized_ak_report.csv.")

Step 7 — Read and Validate Finalized Last.fm A_k
[OK] Connected to Ganache
[INFO] Chain ID: 1337
[INFO] Accounts: 850
[INFO] Contract address: 0x81CC2969f72341fCC3DF58e920BA33CE8DE90f29

[START] Reading finalized Last.fm A_k values from contract...
[INFO] Number of communities: 13
[INFO] Communities: [28, 30, 38, 40, 44, 60, 99, 104, 107, 114, 204, 211, 506]

------------------------------------------------------------------------------------------
[COMMUNITY] 28
expected_Ak           : 1858
onchain_Ak            : 1858
finalized_Ak          : 1858
exists                : True
finalized             : True
confirm_count         : 20
required_count        : 20
confirm_count_matches : True
expired               : False
Ak_matches            : True
required_matches      : True
hash_matches          : True
validation_status     : valid_finalized

------------------------------------------------------------------------------------------
[COMMUNITY] 30
expected_Ak           : 423
onchain_Ak  

### ملخص قياسات الأداء — شغّله بعد اكتمال التثبيت والتحقق النهائي

هذه الخلية تُنتج نفس نوع ملخص Douban، وتتحقق أيضاً من أن القياس اقتصر فعلاً على **13 مجتمعاً / 673 عضواً / 347 معاملة إعلان+تأكيد**.


In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Blockchain Performance Summary — Last.fm (13 final communities)
# =====================================================================================

import numpy as np
import pandas as pd

if not PERF_TX_CSV.exists():
    raise FileNotFoundError(
        f"Performance transaction log not found: {PERF_TX_CSV}. "
        "Run the measured blockchain stages first."
    )

df_perf = pd.read_csv(PERF_TX_CSV)

for col in [
    "gas_used", "latency_sec", "cost_eth", "receipt_status",
    "community_size", "required_confirmations", "stage_elapsed_sec"
]:
    if col in df_perf.columns:
        df_perf[col] = pd.to_numeric(df_perf[col], errors="coerce")

# =====================================================================================
# Stage-level summary
# =====================================================================================

stage_rows = []

for (stage, tx_type), g in df_perf.groupby(
    ["stage", "transaction_type"], dropna=False
):
    total_latency = float(g["latency_sec"].sum())
    successful = int((g["receipt_status"] == 1).sum())

    stage_rows.append({
        "stage": stage,
        "transaction_type": tx_type,
        "transactions": int(len(g)),
        "successful_transactions": successful,
        "total_gas_used": int(g["gas_used"].sum()),
        "mean_gas_used": float(g["gas_used"].mean()),
        "median_gas_used": float(g["gas_used"].median()),
        "min_gas_used": int(g["gas_used"].min()),
        "max_gas_used": int(g["gas_used"].max()),
        "total_latency_sec": total_latency,
        "mean_latency_sec": float(g["latency_sec"].mean()),
        "throughput_tx_per_sec_serial": (
            successful / total_latency if total_latency > 0 else np.nan
        ),
        "total_cost_eth_ganache": float(g["cost_eth"].sum()),
    })

df_stage_summary = pd.DataFrame(stage_rows)
df_stage_summary.to_csv(
    PERF_STAGE_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)

# =====================================================================================
# Authorization/finalization summary by community
# =====================================================================================

auth = df_perf[
    df_perf["stage"].isin(["announcement", "confirmation"])
].copy()

community_rows = []

for community_id, g in auth.groupby("community_id"):
    g_ann = g[g["stage"] == "announcement"]
    g_conf = g[g["stage"] == "confirmation"]

    final_mask = (
        g_conf["is_finalizing"]
        .astype(str)
        .str.lower()
        .isin(["true", "1"])
    )
    g_final = g_conf[final_mask]

    community_size = int(
        pd.to_numeric(g["community_size"], errors="coerce")
        .dropna()
        .max()
    )

    required = int(
        pd.to_numeric(g["required_confirmations"], errors="coerce")
        .dropna()
        .max()
    )

    announcement_latency = float(g_ann["latency_sec"].sum())

    confirmation_stage_elapsed = (
        float(
            pd.to_numeric(
                g_conf["stage_elapsed_sec"],
                errors="coerce"
            ).max()
        )
        if len(g_conf)
        else 0.0
    )

    controlled_processing_time = (
        announcement_latency + confirmation_stage_elapsed
    )

    successful = int((g["receipt_status"] == 1).sum())

    community_rows.append({
        "community_id": int(community_id),
        "community_size": community_size,
        "required_confirmations": required,
        "announcement_transactions": int(len(g_ann)),
        "confirmation_transactions": int(len(g_conf)),
        "authorization_transactions_total": int(len(g)),
        "announcement_gas_used": int(g_ann["gas_used"].sum()),
        "confirmation_gas_used": int(g_conf["gas_used"].sum()),
        "authorization_gas_used_total": int(g["gas_used"].sum()),
        "mean_confirmation_gas_used": (
            float(g_conf["gas_used"].mean())
            if len(g_conf)
            else np.nan
        ),
        "finalizing_confirmation_gas_used": (
            int(g_final["gas_used"].iloc[-1])
            if len(g_final)
            else np.nan
        ),
        "announcement_latency_sec": announcement_latency,
        "mean_confirmation_latency_sec": (
            float(g_conf["latency_sec"].mean())
            if len(g_conf)
            else np.nan
        ),
        "confirmation_stage_elapsed_sec": confirmation_stage_elapsed,
        "controlled_authorization_processing_time_sec": controlled_processing_time,
        "authorization_throughput_tx_per_sec": (
            successful / controlled_processing_time
            if controlled_processing_time > 0
            else np.nan
        ),
        "authorization_cost_eth_ganache": float(g["cost_eth"].sum()),
        "finalization_reached": bool(len(g_final) > 0),
    })

df_community_summary = (
    pd.DataFrame(community_rows)
    .sort_values("community_id")
)

df_community_summary.to_csv(
    PERF_COMMUNITY_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)

# =====================================================================================
# Final-scope checks
# =====================================================================================

actual_communities = sorted(
    df_community_summary["community_id"].astype(int).tolist()
)

actual_member_rows = int(
    df_community_summary["community_size"].sum()
)

actual_authorization_tx = int(
    df_community_summary["authorization_transactions_total"].sum()
)

actual_announcement_tx = int(
    df_community_summary["announcement_transactions"].sum()
)

actual_confirmation_tx = int(
    df_community_summary["confirmation_transactions"].sum()
)

all_finalized = bool(
    df_community_summary["finalization_reached"].all()
)

scope_pass = (
    actual_communities == FINAL_COMMUNITY_IDS
    and actual_member_rows == EXPECTED_FINAL_MEMBER_ROWS
    and actual_authorization_tx == EXPECTED_AUTHORIZATION_TRANSACTIONS
    and actual_announcement_tx == EXPECTED_FINAL_COMMUNITIES
    and actual_confirmation_tx
        == EXPECTED_AUTHORIZATION_TRANSACTIONS - EXPECTED_FINAL_COMMUNITIES
    and all_finalized
)

update_run_metadata(
    completed_at_utc=utc_now_iso(),
    ending_block=(
        int(w3.eth.block_number)
        if w3.is_connected()
        else None
    ),
    transaction_log=str(PERF_TX_CSV),
    stage_summary=str(PERF_STAGE_SUMMARY_CSV),
    community_summary=str(PERF_COMMUNITY_SUMMARY_CSV),
    measured_transactions=int(len(df_perf)),
    final_scope_check=bool(scope_pass),
    measured_final_communities=actual_communities,
    measured_member_rows=actual_member_rows,
    measured_authorization_transactions=actual_authorization_tx,
    measured_announcement_transactions=actual_announcement_tx,
    measured_confirmation_transactions=actual_confirmation_tx,
)

print("=" * 100)
print("[PERFORMANCE SUMMARY — BY STAGE]")
print("=" * 100)
print(df_stage_summary.to_string(index=False))

print("\n" + "=" * 100)
print("[PERFORMANCE SUMMARY — BY COMMUNITY]")
print("=" * 100)
print(df_community_summary.to_string(index=False))

print("\n" + "=" * 100)
print("[FINAL SCOPE CHECK]")
print("=" * 100)
print("Communities                  :", actual_communities)
print("Number of communities        :", len(actual_communities))
print("Aggregate community members  :", actual_member_rows)
print("Announcement transactions    :", actual_announcement_tx)
print("Confirmation transactions    :", actual_confirmation_tx)
print("Authorization transactions   :", actual_authorization_tx)
print("All communities finalized    :", all_finalized)
print("Expected authorization tx    :", EXPECTED_AUTHORIZATION_TRANSACTIONS)
print("Scope check                   :", "PASS" if scope_pass else "FAILED")

if not scope_pass:
    raise RuntimeError(
        "Final Last.fm scope check failed. "
        "Do not use the measurements until the mismatch is resolved."
    )

print("\n[FILES]")
print("Raw transactions :", PERF_TX_CSV)
print("By stage         :", PERF_STAGE_SUMMARY_CSV)
print("By community     :", PERF_COMMUNITY_SUMMARY_CSV)
print("Run metadata     :", PERF_RUN_META_JSON)


### نهاية التشغيل المطلوبة لهذه النسخة

اعتمد التشغيل فقط إذا أعطى ملخص الأداء النهائي:

- Number of communities = **13**
- Aggregate community members = **673**
- Announcement transactions = **13**
- Confirmation transactions = **334**
- Authorization transactions = **347**
- All communities finalized = **True**
- Scope check = **PASS**

هذه هي إعادة Last.fm المتوافقة مع Douban من ناحية العقد الذكي وبيئة القياس.
